# Dạy và học AI tại các trường đại học – Phân tích dữ liệu OpenAlex
**Bài giữa kỳ – Phạm Gia Khánh – B25DCTN064**

Mục lục: 1. Collection → 2. Profiling (đánh giá chất lượng 7 quy tắc) → 3. Cleansing → 4. Structuring → 5. Transformation → 6. Validation (kèm đặc trưng lượng dữ liệu) → 7. Visualization (kèm phần Việt Nam so với thế giới) → 8. Insight.

Notebook chỉ đọc bản chụp dữ liệu thô trong `data/raw/` (tải ngày 23/10/2026 bằng notebook Kaggle; `collect_openalex.py` là bản script của bước tải này để chạy lại), không gọi lại API, nên chạy lại cho cùng kết quả.

In [1]:
import json, glob, os
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import networkx as nx
import pycountry

RAW, OUT = "data/raw", "data/processed"
FIG = f"{OUT}/figs"
os.makedirs(FIG, exist_ok=True)
LOG = json.load(open(f"{RAW}/download_log.json", encoding="utf-8"))
DOWNLOAD_DATE = pd.Timestamp(LOG["ngay_tai"])
API_COUNT = LOG["count_api"]
pio.templates.default = "plotly_white"
pd.set_option("display.max_colwidth", 90)
print("Ngày tải:", DOWNLOAD_DATE.date(), "| filter:", LOG["filter"], "| meta.count:", API_COUNT)

Ngày tải: 2026-10-23 | filter: primary_topic.id:T14414,title_and_abstract.search:"higher education" OR university,type:article|review|conference-paper,publication_year:2015-2026 | meta.count: 23712


## 1. Collection
Nguồn: OpenAlex API (dữ liệu thứ cấp), gọi bằng `requests` với cursor, 200 bài/trang (code tải: `collect_openalex.py`; bản chụp trong `data/raw/` được tải trên Kaggle bằng cùng cách gọi API). Truy vấn F chọn sau khi so 5 biến thể và chấm precision trên mẫu ngẫu nhiên (`query_variants.csv`, `precision_samples.csv`).

Đọc 119 trang JSON. Khi đọc, giữ bản đầu tiên của mỗi ID để kiểm tra trùng (quy tắc 3).

In [2]:
pages = sorted(glob.glob(f"{RAW}/works/page_*.json"))
rows = [(int(os.path.basename(p)[5:9]), w) for p in pages for w in json.load(open(p, encoding="utf-8"))]
first_seen, works_raw, dup_pairs, dup_identical = {}, [], [], 0
for page, w in rows:
    if w["id"] in first_seen:
        first_page, first_w = first_seen[w["id"]]
        dup_pairs.append((first_page, page))
        dup_identical += first_w == w
        continue
    first_seen[w["id"]] = (page, w)
    works_raw.append(w)
n = len(works_raw)
print(f"{len(pages)} trang | {len(rows)} dòng thô | {n} ID duy nhất | {len(dup_pairs)} dòng trùng ID")
print(f"Bản trùng giống hệt bản đầu: {dup_identical}/{len(dup_pairs)}")
print("Trùng nằm ở hai trang liền nhau:", sum(b - a == 1 for a, b in dup_pairs), "| cặp trang nhiều nhất:",
      Counter(dup_pairs).most_common(4))

119 trang | 23699 dòng thô | 23545 ID duy nhất | 154 dòng trùng ID
Bản trùng giống hệt bản đầu: 154/154
Trùng nằm ở hai trang liền nhau: 154 | cặp trang nhiều nhất: [((79, 80), 13), ((85, 86), 12), ((107, 108), 9), ((17, 18), 8)]


**Kết luận:** 154 dòng trùng ID đều giống hệt bản đầu và nằm ở hai trang liền nhau. Đây là hiện tượng phân trang cursor khi OpenAlex cập nhật chỉ mục trong lúc tải, không phải hai bài khác nhau, nên loại ngay khi đọc là đúng. Cùng lý do, khoảng 0,7% bài bị "trượt" qua khe giữa hai trang (xem quy tắc 5).

## 2. Profiling – đánh giá chất lượng theo 7 quy tắc
Tính lại từ JSON thô và so với `quality_scorecard.csv` đã tạo trên Kaggle. Ngưỡng theo bảng D1 của tài liệu hướng dẫn (ngưỡng tự đặt cho bài này).

In [3]:
countries_of = lambda w: {c for a in w.get("authorships", []) for c in (a.get("countries") or [])}
pct = lambda k: round(100 * k / n, 1)
ids = [w["id"] for w in works_raw]
dois = [w["doi"].lower() for w in works_raw if w.get("doi")]
years = pd.Series([w["publication_year"] for w in works_raw])
dates = pd.to_datetime(pd.Series([w["publication_date"] for w in works_raw]))
cites = pd.Series([w.get("cited_by_count") for w in works_raw], dtype="float")
types = Counter(w.get("type") for w in works_raw)
langs = Counter(w.get("language") for w in works_raw)
q1, q3 = cites.quantile([0.25, 0.75]); upper = q3 + 1.5 * (q3 - q1)

prec = pd.read_csv(f"{RAW}/precision_samples.csv")
pF = prec[prec["bien_the"] == "F"]["đúng"]
PRECISION, K = 100 * pF.mean(), len(pF)
MARGIN = 196 * np.sqrt(pF.mean() * (1 - pF.mean()) / K)
print(f"Precision F: {pF.sum()}/{K} = {PRECISION:.0f}% ± {MARGIN:.1f}% | G: {prec[prec.bien_the=='G']['đúng'].mean():.0%}")

def row(rule, metric, value, text, ok=None):
    return {"quy tắc": rule, "chỉ số": metric, "giá trị": value, "ngưỡng": text,
            "kết quả": "—" if ok is None else ("ĐẠT" if ok(value) else "XEM LẠI")}

scorecard = pd.DataFrame([
    row("1. Liên quan", f"Precision mẫu {K} bài ngẫu nhiên (%)", round(PRECISION, 1), ">= 70", lambda v: v >= 70),
    row("2. Đầy đủ", "Thiếu tóm tắt (%)", pct(sum(w.get("abstract_inverted_index") is None for w in works_raw)), "<= 40", lambda v: v <= 40),
    row("2. Đầy đủ", "Không có quốc gia (%)", pct(sum(not countries_of(w) for w in works_raw)), "<= 30", lambda v: v <= 30),
    row("2. Đầy đủ", "Thiếu primary_topic (%)", pct(sum(w.get("primary_topic") is None for w in works_raw)), "= 0", lambda v: v == 0),
    row("2. Đầy đủ", "Thiếu DOI (%)", pct(n - len(dois)), "báo cáo"),
    row("2. Đầy đủ", "Thiếu nhãn ngôn ngữ (%)", pct(langs.get(None, 0)), "báo cáo"),
    row("2. Đầy đủ", "Ngày chỉ có năm (ghi 01-01) (%)", pct(int((dates.dt.strftime("%m-%d") == "01-01").sum())), "báo cáo"),
    row("3. Duy nhất", "Trùng ID khi tải (dòng, đã loại)", len(dup_pairs), "báo cáo"),
    row("3. Duy nhất", "Trùng DOI, khác ID (bài)", len(dois) - len(set(dois)), "= 0", lambda v: v == 0),
    row("4. Hợp lệ", "Năm ngoài 2015–2026 (bài)", int((~years.between(2015, 2026)).sum()), "= 0", lambda v: v == 0),
    row("4. Hợp lệ", "Loại ngoài article/review/conference (bài)",
        sum(v for k, v in types.items() if k not in {"article", "review", "conference-paper"}), "= 0", lambda v: v == 0),
    row("4. Hợp lệ", "Trích dẫn âm hoặc thiếu (bài)", int((cites.isna() | (cites < 0)).sum()), "= 0", lambda v: v == 0),
    row("4. Hợp lệ", "Ngày xuất bản sau ngày tải (bài)", int((dates > DOWNLOAD_DATE).sum()), "báo cáo"),
    row("5. Nhất quán", "Lệch số bài (sau khử trùng) so với API (%)", round(100 * abs(n - API_COUNT) / API_COUNT, 2), "<= 1", lambda v: v <= 1),
    row("6. Đại diện", "Ngôn ngữ khác tiếng Anh, đã xác định (%)", pct(n - langs.get("en", 0) - langs.get(None, 0)), "báo cáo"),
    row("6. Đại diện", "Bài năm 2026, chưa đủ năm (%)", pct(int((years == 2026).sum())), "báo cáo"),
    row("7. Ngoại lai", f"Trích dẫn > {upper:.0f} (Q3 + 1.5·IQR) (%)", pct(int((cites > upper).sum())), "báo cáo"),
])
old = pd.read_csv(f"{RAW}/quality_scorecard.csv")
scorecard["giá trị CSV Kaggle"] = scorecard["chỉ số"].map(dict(zip(old["chỉ số"], old["giá trị"])))
scorecard.to_csv(f"{OUT}/quality_scorecard_verified.csv", index=False)
print("Tổng số bài:", n, "| API:", API_COUNT)
scorecard

Precision F: 96/100 = 96% ± 3.8% | G: 79%


Tổng số bài: 23545 | API: 23712


,quy tắc,chỉ số,giá trị,ngưỡng,kết quả,giá trị CSV Kaggle
0,1. Liên quan,Precision mẫu 100 bài ngẫu nhiên (%),96.0,>= 70,ĐẠT,NaN
1,2. Đầy đủ,Thiếu tóm tắt (%),10.4,<= 40,ĐẠT,10.4
2,2. Đầy đủ,Không có quốc gia (%),26.6,<= 30,ĐẠT,26.6
3,2. Đầy đủ,Thiếu primary_topic (%),0.0,= 0,ĐẠT,0.0
4,2. Đầy đủ,Thiếu DOI (%),10.2,báo cáo,—,10.2
5,2. Đầy đủ,Thiếu nhãn ngôn ngữ (%),29.0,báo cáo,—,NaN
6,2. Đầy đủ,Ngày chỉ có năm (ghi 01-01) (%),14.7,báo cáo,—,NaN
7,3. Duy nhất,"Trùng ID khi tải (dòng, đã loại)",154.0,báo cáo,—,154.0
8,3. Duy nhất,"Trùng DOI, khác ID (bài)",5.0,= 0,XEM LẠI,5.0
9,4. Hợp lệ,Năm ngoài 2015–2026 (bài),0.0,= 0,ĐẠT,0.0


**Đối chiếu với bảng Kaggle:** mọi chỉ số tính lại trùng khớp. Có ba chỗ cần sửa cách gọi tên:
- *Precision* được chấm trên **100** bài ngẫu nhiên (seed = 1), không phải 20: F = 96% ± 3,8%, G = 79%.
- *"Không phải tiếng Anh 35,7%"* của bảng cũ gộp cả bài **thiếu nhãn ngôn ngữ** (29%). Bài có ngôn ngữ khác đã xác định chỉ chiếm 6,6%. Nhãn thiếu dồn vào bài mới (xem D3), nên đó là trễ xử lý của OpenAlex chứ không phải bài không phải tiếng Anh.
- Thêm hai chỉ số bảng cũ không có: 14,7% bài chỉ có **năm** (ngày ghi 01-01), ảnh hưởng chuỗi theo tháng; 5 bài có ngày xuất bản sau ngày tải (bài "in press", năm vẫn hợp lệ).

Mục duy nhất XEM LẠI là 5 cặp trùng DOI, xử lý ở bước Cleansing.

In [4]:
dc = Counter(dois)
pd.DataFrame([{"doi": w["doi"], "id": w["id"].split("/")[-1], "ngày": w["publication_date"], "loại": w["type"],
               "ngôn ngữ": w.get("language"), "trích dẫn": w["cited_by_count"], "tiêu đề": w["title"][:70]}
              for w in works_raw if w.get("doi") and dc[w["doi"].lower()] > 1]).sort_values("doi")

,doi,id,ngày,loại,ngôn ngữ,trích dẫn,tiêu đề
0,https://doi.org/10.1111/bjet.70093,W7219604947,2026-10-04,article,en,0,Multidimensional engagement effects of generative AI adoption among mi
1,https://doi.org/10.1111/bjet.70093,W7214571605,2026-10-04,article,en,0,Multidimensional Engagement Effects of Generative AI Adoption among Mi
2,https://doi.org/10.14456/hrodj.2026.15,W7204853980,2026-08-31,article,NaN,0,The Application of Artificial Intelligence (AI)in the Skill Developmen
9,https://doi.org/10.14456/hrodj.2026.15,W7215243391,2026-07-20,article,th,0,บทบาทของปญญาประดษฐ (AI) ในการพฒนาทกษะอาจารยมหาวทยาลย
6,https://doi.org/10.4230/oasics.slate.2025.1,W4287689466,2025-01-01,conference-paper,en,725,From Prediction to Precision: Leveraging LLMs for Equitable and Data-D
8,https://doi.org/10.4230/oasics.slate.2025.1,W7109983973,2025-01-01,article,en,1,From Prediction to Precision: Leveraging LLMs for Equitable and Data-D
5,https://doi.org/10.6084/m9.figshare.33948028,W7166488794,2025-01-01,conference-paper,en,0,Sustaining rigor and depth in student learning and attainment in the a
7,https://doi.org/10.6084/m9.figshare.33948028,W7213904717,2026-09-21,conference-paper,NaN,0,"Okai-Ugbaje, S. Lenton, D.A. & Smith, J. 2025. Sustaining rigor and de"
3,https://doi.org/10.66857/2446,W7217948587,2026-01-04,article,en,0,Responsible Governance of Generative Artificial Intelligence in Univer
4,https://doi.org/10.66857/2446,W7222882963,2026-10-18,article,en,0,Responsible Governance of Generative Artificial Intelligence in Univer


Năm cặp trùng DOI gồm: bản tiếng Anh và tiếng Thái của cùng một bài, bản figshare đăng lại, bản ghi metadata trùng và một cặp hội thảo/tạp chí (725 so với 1 trích dẫn). Tất cả là **cùng một công trình** → giữ một bản (bản nhiều trích dẫn hơn, rồi bản có tóm tắt và nhãn ngôn ngữ).

### Năm 2026 có nhiều bài hơn 2025 dù mới đến tháng 10 – lỗi hay thật?

In [5]:
yearly = pd.DataFrame(json.load(open(f"{RAW}/yearly.json")))[["key", "count"]].rename(columns={"key": "year"})
yearly["year"] = yearly["year"].astype(int); yearly = yearly.sort_values("year").reset_index(drop=True)
chk = pd.DataFrame({"year": years, "month": dates.dt.month, "yearonly": dates.dt.strftime("%m-%d") == "01-01"})
by_month = chk[~chk.yearonly & chk.year.isin([2025, 2026])].pivot_table(index="month", columns="year", aggfunc="size")
print("Số bài theo năm (API):", dict(zip(yearly.year, yearly["count"])))
print("Bài 2026 có ngày sau ngày tải:", int((dates[years == 2026] > DOWNLOAD_DATE).sum()))
print("Trung bình/tháng (bỏ ngày 01-01), tháng 2–9:", by_month.loc[2:9].mean().round(0).to_dict())
by_month.T

Số bài theo năm (API): {2015: 3, 2016: 9, 2017: 14, 2018: 36, 2019: 78, 2020: 130, 2021: 196, 2022: 218, 2023: 1089, 2024: 3686, 2025: 8078, 2026: 10175}
Bài 2026 có ngày sau ngày tải: 5
Trung bình/tháng (bỏ ngày 01-01), tháng 2–9: {2025: 502.0, 2026: 934.0}


month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2025,264,310,439,428,489,676,566,429,680,620,748,1162
2026,592,628,1063,931,862,1223,945,874,943,638,1,1


**Kết luận:** chỉ 5 bài năm 2026 có ngày sau ngày tải, nên 2026 không bị "thổi phồng" bởi bài in press. Tính theo tháng (bỏ ngày 01-01), mỗi tháng 2026 có khoảng 930 bài, gần gấp đôi khoảng 500 bài/tháng của cùng kỳ 2025: đây là **tăng trưởng thật**. Dù vậy 2026 chưa đủ năm và bài mới chưa kịp được trích dẫn, nên 2026 được loại khỏi YoY/CAGR và vẽ nét đứt.

### D3 – Thiếu dữ liệu có ngẫu nhiên không (MCAR / MAR)?

In [6]:
miss = pd.DataFrame({
    "year": years, "type": [w["type"] for w in works_raw],
    "ngôn_ngữ": [("thiếu nhãn" if w.get("language") is None else "en" if w["language"] == "en" else "khác") for w in works_raw],
    "thiếu_tóm_tắt": [w.get("abstract_inverted_index") is None for w in works_raw],
    "thiếu_quốc_gia": [not countries_of(w) for w in works_raw],
    "thiếu_ngôn_ngữ": [w.get("language") is None for w in works_raw],
})
cols = ["thiếu_tóm_tắt", "thiếu_quốc_gia", "thiếu_ngôn_ngữ"]
(miss.groupby("year")[cols].mean() * 100).round(1).T

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
thiếu_tóm_tắt,100.0,22.2,28.6,13.9,32.1,13.1,16.4,11.6,7.0,8.5,9.6,11.6
thiếu_quốc_gia,100.0,55.6,50.0,38.9,35.9,33.8,32.8,24.5,23.5,24.1,26.2,27.8
thiếu_ngôn_ngữ,0.0,0.0,0.0,0.0,0.0,0.8,0.0,0.9,1.7,2.6,23.9,47.6


In [7]:
display((miss.groupby("type")[cols[:2]].mean() * 100).round(1))
(miss.groupby("ngôn_ngữ")[cols[:2]].mean() * 100).round(1)

,thiếu_tóm_tắt,thiếu_quốc_gia
type,,
article,5.2,27.1
conference-paper,33.3,25.6
review,7.2,15.3


,thiếu_tóm_tắt,thiếu_quốc_gia
ngôn_ngữ,,
en,11.6,22.8
khác,2.2,39.3
thiếu nhãn,9.6,32.0


**Cách đọc:** cả ba kiểu thiếu đều **không** đều nhau giữa các nhóm, nên là **MAR**, không phải MCAR:
- *Thiếu ngôn ngữ* gần 0% đến 2024, rồi 24% năm 2025 và 48% năm 2026 → OpenAlex chưa kịp gán nhãn cho bài mới. Không dùng `language` để lọc.
- *Thiếu tóm tắt* dồn vào **conference-paper** (≈ 33%, so với ≈ 5% ở article) → phân tích chỉ dựa trên tóm tắt sẽ thiên về bài tạp chí. Bài dùng trường `keywords` (99,9% có) cho ⑪ ⑫.
- *Thiếu quốc gia* cao hơn ở bài cũ và bài không phải tiếng Anh (≈ 39%) → bản đồ và network thiên nhẹ về các nước nói tiếng Anh. Ghi vào phần giới hạn.

### D4 – Ngoại lai trích dẫn

In [8]:
pd.DataFrame([{"title": w["title"][:85], "year": w["publication_year"], "type": w["type"], "cites": w["cited_by_count"],
               "fwci": w.get("fwci")} for w in sorted(works_raw, key=lambda w: w["cited_by_count"], reverse=True)[:10]])

,title,year,type,cites,fwci
0,Systematic review of research on artificial intelligence applications in higher educa,2019,review,7126,20.8637
1,"Students’ voices on generative AI: perceptions, benefits, and challenges in higher ed",2023,article,2463,572.5626
2,Exploring the impact of artificial intelligence on teaching and learning in higher ed,2017,article,2009,74.5450
3,What Is the Impact of ChatGPT on Education? A Rapid Review of the Literature,2023,article,1998,465.1199
4,Artificial intelligence in higher education: the state of the field,2023,article,1957,455.1198
5,ChatGPT: Bullshit spewer or the end of traditional assessments in higher education?,2023,article,1803,419.5351
6,A comprehensive AI policy education framework for university teaching and learning,2023,article,1556,361.3982
7,Intelligence Unleashed: An argument for AI in Education,2016,article,1410,50.2635
8,Embracing the future of Artificial Intelligence in the classroom: the relevance of AI,2024,article,1099,413.8091
9,New Era of Artificial Intelligence in Education: Towards a Sustainable Multifaceted R,2023,article,1080,251.3955


Mười bài trích dẫn cao nhất đều đúng chủ đề: tổng quan hệ thống, bài nổi tiếng về ChatGPT và đánh giá trong giáo dục đại học. Đây là **ngoại lai thật** (slide 19, kiến thức chuyên môn): giữ lại, vẽ bằng thang log, so sánh bằng trung vị và FWCI thay vì trung bình.

## 3. Cleansing
Áp dụng `cleaning_plan.csv` (đã cập nhật theo kết quả Profiling): ép kiểu, khử trùng DOI, ghép tóm tắt, chuẩn hoá mã quốc gia.

In [9]:
def rebuild_abstract(inv):
    if not inv:
        return None
    pos = {i: word for word, idx in inv.items() for i in idx}
    return " ".join(pos[i] for i in sorted(pos))

def iso3(c):
    r = pycountry.countries.get(alpha_2=c)
    return r.alpha_3 if r else {"XK": "XKX"}.get(c)

df = pd.DataFrame([{
    "id": w["id"].split("/")[-1],
    "doi": w["doi"].lower() if w.get("doi") else None,
    "title": w["title"],
    "year": w["publication_year"],
    "date": w["publication_date"],
    "type": w["type"],
    "language": w.get("language"),
    "cites": w["cited_by_count"],
    "fwci": w.get("fwci"),
    "ref_count": w.get("referenced_works_count"),
    "is_oa": (w.get("open_access") or {}).get("is_oa"),
    "abstract": rebuild_abstract(w.get("abstract_inverted_index")),
    "keywords": [k["display_name"] for k in (w.get("keywords") or [])],
    "n_authors": len(w.get("authorships", [])),
    "countries": sorted(countries_of(w)),
    "institutions": sorted({i["id"].split("/")[-1] for a in w.get("authorships", []) for i in a.get("institutions", [])}),
} for w in works_raw])

df["date"] = pd.to_datetime(df["date"])
df = df.astype({"year": "int16", "cites": "int32", "fwci": "float64", "ref_count": "int32", "n_authors": "int16",
                "type": "category", "is_oa": "bool"})
df["has_abstract"] = df["abstract"].notna()
df["date_is_year_only"] = df["date"].dt.strftime("%m-%d") == "01-01"

before = len(df)
ranked = df.assign(_lang=df["language"].notna()).sort_values(
    ["cites", "has_abstract", "_lang", "date"], ascending=[False, False, False, True])
dropped_doi = ranked[ranked["doi"].notna() & ranked.duplicated("doi")]["id"].tolist()
df = df[~df["id"].isin(dropped_doi)].sort_values("date").reset_index(drop=True)
print(f"Khử trùng DOI: {before} → {len(df)} bài, loại {dropped_doi}")

all_cc = sorted({c for cs in df["countries"] for c in cs})
unmapped = [c for c in all_cc if iso3(c) is None]
print(f"{len(all_cc)} mã quốc gia, không đổi được sang ISO-3: {unmapped}")
assert df["id"].is_unique and df.loc[df.doi.notna(), "doi"].is_unique

Khử trùng DOI: 23545 → 23540 bài, loại ['W7109983973', 'W7214571605', 'W7213904717', 'W7222882963', 'W7204853980']
165 mã quốc gia, không đổi được sang ISO-3: []


## 4. Structuring – các bảng dạng dài (tidy)
Một bài có nhiều quốc gia, trường và từ khoá, nên tách thành bảng dài: mỗi dòng một cặp (bài, quốc gia), (bài, trường), (bài, từ khoá).

In [10]:
CONTINENTS = {
    "Châu Á": "AF AM AZ BH BD BT BN KH CN CY GE HK IN ID IR IQ IL JP JO KZ KW KG LA LB MO MY MV MN MM NP KP OM PK PS PH QA SA SG KR LK SY TW TJ TH TL TR TM AE UZ VN YE",
    "Châu Âu": "AL AD AT BY BE BA BG HR CZ DK EE FO FI FR DE GI GR HU IS IE IM IT XK LV LI LT LU MT MD MC ME NL MK NO PL PT RO RU SM RS SK SI ES SE CH UA GB VA GG JE AX",
    "Bắc Mỹ": "AG BS BB BZ CA CR CU DM DO SV GD GT HT HN JM MX NI PA KN LC VC TT US PR BM KY GL VI VG AW CW SX MQ GP",
    "Nam Mỹ": "AR BO BR CL CO EC GY PY PE SR UY VE GF FK",
    "Châu Phi": "DZ AO BJ BW BF BI CV CM CF TD KM CD CG CI DJ EG GQ ER SZ ET GA GM GH GN GW KE LS LR LY MG MW ML MR MU YT MA MZ NA NE NG RE RW SH ST SN SC SL SO ZA SS SD TZ TG TN UG EH ZM ZW",
    "Châu Đại Dương": "AS AU CK FJ PF GU KI MH FM NR NC NZ NU PW PG WS SB TO TV VU NF",
}
CONT_OF = {c: k for k, v in CONTINENTS.items() for c in v.split()}
COLOR_CONT = {"Châu Á": "#0072B2", "Châu Âu": "#E69F00", "Bắc Mỹ": "#009E73", "Nam Mỹ": "#CC79A7",
              "Châu Phi": "#D55E00", "Châu Đại Dương": "#56B4E9", "Khác": "#999999"}
print("Mã chưa có châu lục:", [c for c in all_cc if c not in CONT_OF])

work_country = df[["id", "year", "countries"]].explode("countries").dropna().rename(columns={"countries": "country"})
work_country["iso3"] = work_country["country"].map(iso3)
work_country["continent"] = work_country["country"].map(CONT_OF).fillna("Khác")
work_inst = df[["id", "year", "cites", "institutions"]].explode("institutions").dropna().rename(columns={"institutions": "inst"})
work_kw = df[["id", "year", "keywords"]].explode("keywords").dropna().rename(columns={"keywords": "keyword"})
print(f"work_country {len(work_country)} dòng | work_inst {len(work_inst)} | work_kw {len(work_kw)}")

country_year = pd.DataFrame(json.load(open(f"{RAW}/country_year.json", encoding="utf-8")))
country_year["iso3"] = country_year["country"].map(iso3)
country_year["continent"] = country_year["country"].map(CONT_OF).fillna("Khác")
insts = pd.json_normalize(json.load(open(f"{RAW}/institutions.json", encoding="utf-8")))
insts["id"] = insts["id"].str.split("/").str[-1]
multi = work_country.groupby("id").size()
print(f"Lượt tham gia quốc gia (group_by, toàn bộ): {country_year['count'].sum():,} — khác số bài {API_COUNT:,} vì "
      f"{(multi > 1).sum():,} bài có nhiều nước (đếm ở mỗi nước) và {(df.countries.str.len() == 0).sum():,} bài không có nước nào (số trong dữ liệu tải về)")

Mã chưa có châu lục: []
work_country 20820 dòng | work_inst 28224 | work_kw 217262
Lượt tham gia quốc gia (group_by, toàn bộ): 20,970 — khác số bài 23,712 vì 2,585 bài có nhiều nước (đếm ở mỗi nước) và 6,255 bài không có nước nào (số trong dữ liệu tải về)


## 5. Transformation – đặc trưng mới và chỉ số xu hướng
- `log_cit = log1p(cites)`: trích dẫn lệch phải rất mạnh (trung vị 0–1, tối đa hàng nghìn); Min-Max không sửa được độ lệch, còn log thì có.
- `citations_per_year = cites / max(tuổi bài, 1 năm)`: giảm lợi thế của bài cũ; mẫu số tối thiểu 1 năm để bài vài tuần tuổi không bị phóng đại.
- `period`: 2015–2022 (trước ChatGPT, ra mắt 30/11/2022) và 2023–2026.

In [11]:
age = ((DOWNLOAD_DATE - df["date"]).dt.days / 365.25).clip(lower=1)
continent_of_work = df["countries"].map(lambda cs: {CONT_OF.get(c, "Khác") for c in cs})
df = df.assign(
    log_cit=np.log1p(df["cites"]),
    age_years=age.round(2),
    citations_per_year=(df["cites"] / age).round(3),
    n_countries=df["countries"].str.len().astype("int16"),
    n_institutions=df["institutions"].str.len().astype("int16"),
    is_international=df["countries"].str.len() > 1,
    period=pd.Categorical(np.where(df["year"] <= 2022, "2015–2022", "2023–2026")),
    continent=continent_of_work.map(lambda s: "Không rõ" if not s else next(iter(s)) if len(s) == 1 else "Nhiều châu lục"),
)
df[["log_cit", "citations_per_year", "n_authors", "n_countries", "is_international", "period", "continent"]].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
log_cit,23540.0,NaN,NaN,NaN,0.662854,1.156945,0.0,0.0,0.0,1.098612,8.871646
citations_per_year,23540.0,NaN,NaN,NaN,3.029517,17.008905,0.0,0.0,0.0,1.0,1019.495
n_authors,23540.0,NaN,NaN,NaN,2.720476,2.165355,0.0,1.0,2.0,4.0,100.0
n_countries,23540.0,NaN,NaN,NaN,0.884452,0.803305,0.0,0.0,1.0,1.0,45.0
is_international,23540,2,False,20955,NaN,NaN,NaN,NaN,NaN,NaN,NaN
period,23540,2,2023–2026,22859,NaN,NaN,NaN,NaN,NaN,NaN,NaN
continent,23540,8,Châu Á,8547,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
yr = yearly.copy()
yr["yoy_%"] = (yr["count"].pct_change() * 100).round(1)
cagr = lambda a, b: (yr.set_index("year").loc[b, "count"] / yr.set_index("year").loc[a, "count"]) ** (1 / (b - a)) - 1
CAGR_PRE, CAGR_POST = cagr(2018, 2022), cagr(2022, 2025)
YOY_STD = yr.loc[yr.year.between(2019, 2025), "yoy_%"].std()
print(f"CAGR 2018–2022: {CAGR_PRE:.0%} | CAGR 2022–2025: {CAGR_POST:.0%} | độ lệch chuẩn YoY 2019–2025: {YOY_STD:.0f} điểm %")
yr

CAGR 2018–2022: 57% | CAGR 2022–2025: 233% | độ lệch chuẩn YoY 2019–2025: 134 điểm %


,year,count,yoy_%
0,2015,3,NaN
1,2016,9,200.0
2,2017,14,55.6
3,2018,36,157.1
4,2019,78,116.7
5,2020,130,66.7
6,2021,196,50.8
7,2022,218,11.2
8,2023,1089,399.5
9,2024,3686,238.5


In [13]:
LAST_FULL_MONTH = (DOWNLOAD_DATE.to_period("M") - 1).to_timestamp()
mon = (df[~df["date_is_year_only"] & (df["date"] >= "2019-01-01") & (df["date"] < DOWNLOAD_DATE.to_period("M").to_timestamp())]
       .set_index("date").resample("MS").size().rename("count").to_frame())
mon["rolling3"] = mon["count"].rolling(3).mean().round(1)
mon["mom_%"] = (mon["count"].pct_change() * 100).round(1)
excluded_month_share = 100 * df["date_is_year_only"].mean()
print(f"Bỏ {excluded_month_share:.1f}% bài chỉ có năm khỏi chuỗi tháng; tháng cuối đủ dữ liệu: {LAST_FULL_MONTH:%m/%Y}")
spikes = mon[mon.index >= "2023-01-01"].sort_values("mom_%", key=abs, ascending=False).head(5)
spikes

Bỏ 14.7% bài chỉ có năm khỏi chuỗi tháng; tháng cuối đủ dữ liệu: 09/2026


,count,rolling3,mom_%
date,,,
2023-03-01,38,23.3,137.5
2026-03-01,1063,761.0,69.3
2025-09-01,680,558.3,58.5
2023-12-01,199,143.0,57.9
2025-12-01,1162,843.3,55.3


Tháng 1 và tháng 12 tăng vọt lặp lại mỗi năm. Một phần là bài chỉ có năm (đã loại), phần còn lại là bài thuộc số tạp chí tháng 1 hoặc cuối năm. Đó là nhịp xuất bản, không phải biến động của sự quan tâm, nên đọc đường trung bình trượt 3 tháng là chính.

In [14]:
DATA_DICT = pd.DataFrame([
    ("id", "OpenAlex work ID (W…)", "Khoá chính; khử trùng khi đọc"),
    ("doi", "DOI chữ thường", "Thiếu 10%: giữ; khử trùng 5 cặp"),
    ("title", "Tiêu đề", "Không thiếu"),
    ("year", "Năm xuất bản", "2015–2026; 2026 chưa đủ năm"),
    ("date", "Ngày xuất bản", "Ép datetime; 01-01 coi là chỉ có năm"),
    ("type", "article / review / conference-paper", "Category"),
    ("language", "Mã ngôn ngữ ISO-639", "Thiếu nhiều ở 2025–2026 (MAR): không dùng để lọc"),
    ("cites", "Số trích dẫn", "Ngoại lai thật: giữ, dùng log"),
    ("fwci", "Field-Weighted Citation Impact", "Chuẩn hoá theo năm và lĩnh vực"),
    ("ref_count", "Số tài liệu tham khảo", ""),
    ("is_oa", "Truy cập mở", ""),
    ("abstract", "Tóm tắt ghép từ inverted index", "Thiếu: pairwise deletion"),
    ("keywords", "Danh sách từ khoá OpenAlex", "Dùng cho ⑪ ⑫"),
    ("n_authors / n_institutions / n_countries", "Đếm từ authorships", "0 khi không có thông tin"),
    ("countries", "Mã ISO-2 của tác giả", "Không có: loại khỏi bản đồ/network"),
    ("is_international", "Có ≥ 2 quốc gia", "Đặc trưng mới"),
    ("log_cit", "log1p(cites)", "Đặc trưng mới"),
    ("citations_per_year", "cites / max(tuổi, 1)", "Đặc trưng mới"),
    ("age_years", "Tuổi bài tính đến ngày tải (≥ 1)", "Đặc trưng mới"),
    ("period", "2015–2022 / 2023–2026", "Đặc trưng mới"),
    ("continent", "Châu lục của tác giả", "Nhiều châu lục / Không rõ"),
], columns=["cột", "ý nghĩa", "xử lý"])
DATA_DICT["kiểu"] = DATA_DICT["cột"].map(lambda c: str(df[c].dtype) if c in df else "int16")
DATA_DICT["% thiếu"] = DATA_DICT["cột"].map(lambda c: round(100 * df[c].isna().mean(), 1) if c in df else 0.0)
DATA_DICT.to_csv(f"{OUT}/data_dictionary.csv", index=False)
DATA_DICT

,cột,ý nghĩa,xử lý,kiểu,% thiếu
0,id,OpenAlex work ID (W…),Khoá chính; khử trùng khi đọc,str,0.0
1,doi,DOI chữ thường,Thiếu 10%: giữ; khử trùng 5 cặp,str,10.2
2,title,Tiêu đề,Không thiếu,str,0.0
3,year,Năm xuất bản,2015–2026; 2026 chưa đủ năm,int16,0.0
4,date,Ngày xuất bản,Ép datetime; 01-01 coi là chỉ có năm,datetime64[us],0.0
5,type,article / review / conference-paper,Category,category,0.0
6,language,Mã ngôn ngữ ISO-639,Thiếu nhiều ở 2025–2026 (MAR): không dùng để lọc,str,29.0
7,cites,Số trích dẫn,"Ngoại lai thật: giữ, dùng log",int32,0.0
8,fwci,Field-Weighted Citation Impact,Chuẩn hoá theo năm và lĩnh vực,float64,0.0
9,ref_count,Số tài liệu tham khảo,,int32,0.0


## 6. Validation
Kiểm tra lại các ràng buộc sau khi làm sạch trước khi vẽ và lưu dữ liệu đã xử lý.

In [15]:
checks = {
    "ID duy nhất": df["id"].is_unique,
    "DOI duy nhất": df.loc[df.doi.notna(), "doi"].is_unique,
    "Năm trong 2015–2026": df["year"].between(2015, 2026).all(),
    "Năm khớp ngày": (df["date"].dt.year == df["year"]).all(),
    "Trích dẫn ≥ 0": (df["cites"] >= 0).all(),
    "Đúng 3 loại tài liệu": set(df["type"].unique()) <= {"article", "review", "conference-paper"},
    "Lệch so với API ≤ 1%": abs(len(df) - API_COUNT) / API_COUNT <= 0.01,
    "Mọi mã quốc gia có ISO-3": work_country["iso3"].notna().all(),
    "Tổng yearly.json = meta.count": yearly["count"].sum() == API_COUNT,
}
assert all(checks.values()), checks
df.drop(columns=["abstract", "keywords", "countries", "institutions"]).to_csv(f"{OUT}/works_clean.csv", index=False)
df[["id", "abstract"]].dropna().to_csv(f"{OUT}/abstracts.csv", index=False)
work_country.to_csv(f"{OUT}/work_country.csv", index=False)
work_inst.to_csv(f"{OUT}/work_institution.csv", index=False)
work_kw.to_csv(f"{OUT}/work_keyword.csv", index=False)
pd.Series(checks, name="đạt").to_frame()

,đạt
ID duy nhất,True
DOI duy nhất,True
Năm trong 2015–2026,True
Năm khớp ngày,True
Trích dẫn ≥ 0,True
Đúng 3 loại tài liệu,True
Lệch so với API ≤ 1%,True
Mọi mã quốc gia có ISO-3,True
Tổng yearly.json = meta.count,True


### Đặc trưng chính về lượng dữ liệu
Tổng hợp quy mô dữ liệu ở từng bước, từ khi tải đến sau làm sạch. Mọi con số được tính lại từ `data/raw/` (gồm `download_log.json` và `cursor_state.json`) và lưu ở `data_volume.csv`, để báo cáo đọc lại mà không phải gõ tay.


In [16]:
raw_files = [p for p in glob.glob(f"{RAW}/**/*", recursive=True) if os.path.isfile(p)]
CURSOR = json.load(open(f"{RAW}/cursor_state.json", encoding="utf-8"))
assert CURSOR["cursor"] is None and CURSOR["page"] == len(pages), CURSOR  # cursor rỗng: đã tải đến trang cuối
clean_ids = set("https://openalex.org/" + df["id"])
n_unique_authors = len({a["author"]["id"] for w in works_raw if w["id"] in clean_ids for a in w["authorships"] if a.get("author", {}).get("id")})
miss_pct = lambda col: round(100 * df[col].isna().mean(), 1)
VOLUME = pd.DataFrame([
    ("Thu thập", "Số bài OpenAlex báo khớp truy vấn (meta.count)", API_COUNT),
    ("Thu thập", "Số trang đã tải bằng cursor (200 bài/trang)", len(pages)),
    ("Thu thập", "Số dòng bài báo thô", len(rows)),
    ("Thu thập", "Dung lượng dữ liệu thô trong data/raw (MB)", round(sum(map(os.path.getsize, raw_files)) / 1e6, 1)),
    ("Thu thập", "Trong đó 119 trang JSON bài báo (MB)", round(sum(map(os.path.getsize, pages)) / 1e6, 1)),
    ("Làm sạch", "Dòng trùng ID đã loại", len(dup_pairs)),
    ("Làm sạch", "Bài trùng DOI đã loại", len(dropped_doi)),
    ("Làm sạch", "Số bài sau làm sạch", len(df)),
    ("Làm sạch", "Tỷ lệ dòng thô bị loại (%)", round(100 * (len(rows) - len(df)) / len(rows), 2)),
    ("Phạm vi", "Khoảng thời gian", f"{df['year'].min()}–{df['year'].max()}"),
    ("Phạm vi", "Số quốc gia có tác giả", int(country_year["country"].nunique())),
    ("Phạm vi", "Số trường/viện khác nhau (ID OpenAlex)", int(work_inst["inst"].nunique())),
    ("Phạm vi", "Số trường được tải thêm hồ sơ", len(insts)),
    ("Phạm vi", "Số tác giả khác nhau", n_unique_authors),
    ("Phạm vi", "Số từ khoá khác nhau", int(work_kw["keyword"].nunique())),
    ("Thiếu dữ liệu", "Thiếu tóm tắt (%)", miss_pct("abstract")),
    ("Thiếu dữ liệu", "Không có quốc gia (%)", round(100 * (df["countries"].str.len() == 0).mean(), 1)),
    ("Thiếu dữ liệu", "Thiếu nhãn ngôn ngữ (%)", miss_pct("language")),
    ("Thiếu dữ liệu", "Thiếu DOI (%)", miss_pct("doi")),
    ("Bảng dài", "Dòng (bài, quốc gia) / (bài, trường) / (bài, từ khoá)", f"{len(work_country):,} / {len(work_inst):,} / {len(work_kw):,}".replace(",", ".")),
], columns=["nhóm", "chỉ số", "giá trị"])
VOLUME.to_csv(f"{OUT}/data_volume.csv", index=False)
VOLUME


,nhóm,chỉ số,giá trị
0,Thu thập,Số bài OpenAlex báo khớp truy vấn (meta.count),23712
1,Thu thập,Số trang đã tải bằng cursor (200 bài/trang),119
2,Thu thập,Số dòng bài báo thô,23699
3,Thu thập,Dung lượng dữ liệu thô trong data/raw (MB),161.8
4,Thu thập,Trong đó 119 trang JSON bài báo (MB),161.6
5,Làm sạch,Dòng trùng ID đã loại,154
6,Làm sạch,Bài trùng DOI đã loại,5
7,Làm sạch,Số bài sau làm sạch,23540
8,Làm sạch,Tỷ lệ dòng thô bị loại (%),0.67
9,Phạm vi,Khoảng thời gian,2015–2026


## 7. Visualization – 16 biểu đồ tương tác (Plotly)
Mỗi biểu đồ lưu thành JSON trong `data/processed/figs/` kèm tiêu đề nêu kết luận và dòng "Cách đọc"; `build_dashboard.py` ghép chúng thành `index.html`. Bảng dữ liệu dạng dài của từng biểu đồ lưu thành `chartNN_*.csv`.

In [17]:
FONT = dict(size=13)
MANIFEST = []
FMT = lambda x: f"{x:,.0f}".replace(",", ".")
VN = lambda x, d=1: f"{x:,.{d}f}".replace(",", "_").replace(".", ",").replace("_", ".")
VI_NAME = {"CN": "Trung Quốc", "US": "Hoa Kỳ", "ID": "Indonesia", "GB": "Anh", "IN": "Ấn Độ", "ES": "Tây Ban Nha",
           "AU": "Úc", "SA": "Ả Rập Xê Út", "MY": "Malaysia", "PK": "Pakistan", "RU": "Nga", "VN": "Việt Nam", "DE": "Đức",
           "TR": "Thổ Nhĩ Kỳ", "PH": "Philippines", "KR": "Hàn Quốc", "JP": "Nhật Bản", "CA": "Canada", "BR": "Brazil"}

def save(fig, num, section, title, how, table=None, name=""):
    fig.update_layout(title=dict(text=title, font=dict(size=17)), font=FONT, margin=dict(l=60, r=30, t=70, b=50),
                      hoverlabel=dict(font_size=13))
    path = f"{FIG}/fig{num:02d}.json"
    fig.write_json(path)
    if table is not None:
        table.to_csv(f"{OUT}/chart{num:02d}_{name}.csv", index=False)
    MANIFEST.append({"num": num, "section": section, "file": os.path.basename(path), "title": title, "how": how})
    return fig

### Xu hướng

In [18]:
c = yr.set_index("year")["count"]
mult = c[2025] / c[2022]
t1 = yr.assign(kind=np.where(yr.year <= 2025, "Đủ năm", "2026 (đến 23/10)"))
fig1 = go.Figure()
fig1.add_scatter(x=yr.year[yr.year <= 2025], y=c[c.index <= 2025], mode="lines+markers", name="Đủ năm",
                 line=dict(color="#0072B2", width=3), hovertemplate="%{x}: %{y:,} bài<extra></extra>")
fig1.add_scatter(x=[2025, 2026], y=[c[2025], c[2026]], mode="lines+markers", name="2026 (đến 23/10, chưa đủ năm)",
                 line=dict(color="#0072B2", width=3, dash="dash"), hovertemplate="%{x}: %{y:,} bài<extra></extra>")
fig1.add_vline(x=2022.91, line_dash="dot", line_color="#555")
fig1.add_annotation(x=2022.91, y=c[2025] * 0.9, text="ChatGPT ra mắt<br>30/11/2022", showarrow=False, xanchor="right")
fig1.update_xaxes(title="Năm", rangeslider_visible=True, dtick=1)
fig1.update_yaxes(title="Số bài", rangemode="tozero", separatethousands=True)
fig1.update_layout(separators=",.", legend=dict(orientation="h", y=1.02, x=0))
save(fig1, 1, "xu-huong",
     f"Số bài năm 2025 gấp {mult:.0f} lần năm 2022" + ("; riêng 10 tháng 2026 đã vượt cả năm 2025" if c[2026] > c[2025] else ""),
     f"Mỗi điểm là số bài của một năm, do OpenAlex đếm trên toàn bộ {FMT(API_COUNT)} bài. Đoạn nét đứt là 2026 chưa đủ năm. Kéo thanh trượt bên dưới để phóng to một giai đoạn.",
     t1, "bai_theo_nam")

In [19]:
cy = country_year[country_year.year >= 2019].groupby(["year", "continent"], as_index=False)["count"].sum()
share = cy.assign(pct=100 * cy["count"] / cy.groupby("year")["count"].transform("sum"))
s = share.set_index(["year", "continent"])["pct"]
fig2 = px.area(cy, x="year", y="count", color="continent", groupnorm="percent", color_discrete_map=COLOR_CONT,
               category_orders={"continent": list(COLOR_CONT)}, labels={"year": "Năm", "count": "Lượt tham gia", "continent": "Châu lục"})
fig2.update_traces(hovertemplate="%{fullData.name}: %{y:.1f}%<extra></extra>")
fig2.update_yaxes(title="Tỷ trọng lượt tham gia (%)", ticksuffix="%")
fig2.update_xaxes(dtick=1)
fig2.update_layout(hovermode="x unified")
save(fig2, 2, "xu-huong",
     f"Châu Á chiếm {s[(2026, 'Châu Á')]:.0f}% lượt tham gia năm 2026, tăng từ {s[(2019, 'Châu Á')]:.0f}% năm 2019",
     "Mỗi dải là phần trăm lượt tham gia của một châu lục trong năm (một bài có tác giả ở hai nước được tính cho cả hai). Bắt đầu từ 2019 vì các năm trước có dưới 40 bài. Bấm vào chú thích để ẩn/hiện châu lục.",
     share, "ty_trong_chau_luc")

### Biến động

In [20]:
m = mon.reset_index()
fig3 = go.Figure()
fig3.add_bar(x=m["date"], y=m["count"], name="Số bài/tháng", marker_color="#9ecae1",
             hovertemplate="%{x|%m/%Y}: %{y:,} bài<extra></extra>")
fig3.add_scatter(x=m["date"], y=m["rolling3"], name="Trung bình trượt 3 tháng", line=dict(color="#0072B2", width=3),
                 hovertemplate="%{x|%m/%Y}: %{y:,.0f}<extra></extra>")
fig3.add_vline(x=pd.Timestamp("2022-11-30").timestamp() * 1000, line_dash="dot", line_color="#555")
fig3.update_xaxes(title="Tháng", rangeslider_visible=True)
fig3.update_yaxes(title="Số bài", rangemode="tozero")
fig3.update_layout(separators=",.", legend=dict(orientation="h", y=1.02, x=0))
r = mon["rolling3"]
save(fig3, 3, "bien-dong",
     f"Trung bình trượt tăng từ {r['2022-12-01']:.0f} lên {r[LAST_FULL_MONTH]:.0f} bài/tháng kể từ khi ChatGPT ra mắt",
     f"Cột là số bài từng tháng, đường là trung bình 3 tháng gần nhất. Bỏ {excluded_month_share:.0f}% bài chỉ ghi năm xuất bản (ngày 01-01) và tháng 10/2026 chưa hết. Đỉnh tháng 1 và tháng 12 là nhịp ra số tạp chí.",
     m, "bai_theo_thang")

In [21]:
y4 = yr[yr.year.between(2019, 2025)].copy()
y4["chiều"] = np.where(y4["yoy_%"] >= 0, "Tăng", "Giảm")
fig4 = px.bar(y4, x="year", y="yoy_%", color="chiều", color_discrete_map={"Tăng": "#0072B2", "Giảm": "#D55E00"},
              text=y4["yoy_%"].map(lambda v: f"{v:+.0f}%"), labels={"year": "Năm", "yoy_%": "Tăng so với năm trước (%)"})
fig4.update_traces(textposition="outside", hovertemplate="%{x}: %{y:+.1f}%<extra></extra>")
fig4.update_yaxes(ticksuffix="%", rangemode="tozero")
fig4.update_xaxes(dtick=1)
top_yoy = y4.loc[y4["yoy_%"].idxmax()]
save(fig4, 4, "bien-dong",
     f"Năm {top_yoy.year:.0f} tăng mạnh nhất (+{top_yoy['yoy_%']:.0f}%); sau đó tốc độ chậm dần nhưng vẫn từ +{y4.loc[y4.year > top_yoy.year, 'yoy_%'].min():.0f}% mỗi năm",
     f"Mỗi cột là phần trăm thay đổi số bài so với năm trước. 2026 chưa đủ năm nên không đưa vào. Độ lệch chuẩn YoY 2019–2025 là {YOY_STD:.0f} điểm phần trăm: tăng trưởng rất không đều.",
     y4, "yoy")

### Tác động

In [22]:
TICKS = [0, 1, 3, 10, 30, 100, 300, 1000, 3000]
COLOR_PERIOD = {"2015–2022": "#0072B2", "2023–2026": "#E69F00"}
med = df.groupby("period", observed=True)["citations_per_year"].median()
cited = 100 * df.groupby("period", observed=True)["cites"].apply(lambda x: (x > 0).mean())
fig5 = px.violin(df.assign(y=np.log1p(df["citations_per_year"])), x="period", y="y", color="period", box=True, points="outliers",
                 color_discrete_map=COLOR_PERIOD, hover_data={"citations_per_year": ":.1f", "y": False, "period": False},
                 labels={"period": "Giai đoạn", "y": "Trích dẫn mỗi năm (thang log)"})
fig5.update_yaxes(tickvals=np.log1p(TICKS[:7]), ticktext=[str(t) for t in TICKS[:7]])
fig5.update_traces(hoveron="points", hovertemplate="%{customdata[0]:.1f} trích dẫn/năm<extra></extra>")
fig5.update_layout(showlegend=False)
save(fig5, 5, "tac-dong",
     f"Bài từ 2023 phần lớn chưa có trích dẫn: trung vị {VN(med['2023–2026'])} lần/năm so với {VN(med['2015–2022'])} của bài trước 2023",
     f"Độ rộng của violin cho biết bao nhiêu bài có mức trích dẫn đó; hộp ở giữa là tứ phân vị, vạch ngang là trung vị. Trục dọc dùng thang log. Chỉ {cited['2023–2026']:.0f}% bài từ 2023 đã có trích dẫn, so với {cited['2015–2022']:.0f}% bài trước đó. Bài mới chưa đủ thời gian để được trích dẫn, nên khoảng cách này một phần là do độ trễ.",
     df[["id", "period", "citations_per_year", "fwci"]], "trich_dan_giai_doan")

In [23]:
zero_share = 100 * (df["cites"] == 0).mean()
fig6 = px.histogram(df, x="log_cit", color="period", nbins=40, barmode="overlay", opacity=0.7, histnorm="percent",
                    color_discrete_map=COLOR_PERIOD, labels={"log_cit": "Số trích dẫn (thang log)", "period": "Giai đoạn"})
fig6.update_xaxes(tickvals=np.log1p(TICKS), ticktext=[str(t) for t in TICKS])
fig6.update_yaxes(title="Tỷ lệ bài trong giai đoạn (%)", ticksuffix="%")
fig6.update_traces(hovertemplate="%{y:.1f}% số bài<extra></extra>")
save(fig6, 6, "tac-dong",
     f"{zero_share:.0f}% số bài chưa có trích dẫn nào; trích dẫn tập trung vào một số ít bài",
     "Mỗi cột là phần trăm số bài của giai đoạn rơi vào một khoảng trích dẫn; trục ngang dùng thang log nên 0, 1, 10, 100 cách đều nhau. Hai màu chồng lên nhau để so hình dạng phân phối.",
     df[["id", "period", "cites", "log_cit"]], "phan_phoi_trich_dan")

### Quan hệ

In [24]:
inst_stats = (work_inst.groupby("inst").agg(so_bai=("id", "nunique"), trich_dan=("cites", "sum")).reset_index()
              .merge(insts[["id", "display_name", "country_code"]], left_on="inst", right_on="id"))
inst_stats["continent"] = inst_stats["country_code"].map(CONT_OF).fillna("Khác")
inst_plot = inst_stats[inst_stats["trich_dan"] > 0]
fig7 = px.scatter(inst_plot, x="so_bai", y="trich_dan", color="continent", hover_name="display_name",
                  hover_data={"country_code": True, "continent": False}, color_discrete_map=COLOR_CONT,
                  log_x=True, log_y=True, trendline="ols", trendline_scope="overall", trendline_color_override="#333",
                  trendline_options=dict(log_x=True, log_y=True), category_orders={"continent": list(COLOR_CONT)},
                  labels={"so_bai": "Số bài về chủ đề (thang log)", "trich_dan": "Tổng trích dẫn (thang log)", "continent": "Châu lục"})
ols = px.get_trendline_results(fig7).px_fit_results.iloc[0]
fig7.update_traces(selector=dict(mode="lines"), name="Hồi quy OLS (log–log)")
SLOPE, R2 = ols.params[1], ols.rsquared
save(fig7, 7, "quan-he",
     f"Trường viết nhiều hơn 10 lần được trích dẫn nhiều hơn khoảng {10 ** SLOPE:.0f} lần (độ dốc log–log {VN(SLOPE, 2)}, R² = {VN(R2, 2)})",
     f"Mỗi chấm là một trong {len(inst_plot)} trường có nhiều bài nhất (bỏ {len(inst_stats) - len(inst_plot)} trường chưa có trích dẫn). Đường đen là hồi quy OLS trên log của hai trục. Độ dốc lớn hơn 1 nghĩa là trích dẫn tăng nhanh hơn số bài; R² chỉ {VN(R2, 2)} nghĩa là ở cùng số bài, trích dẫn giữa các trường vẫn chênh nhau rất xa. Rê chuột để xem tên trường.",
     inst_stats, "truong")

In [25]:
num = df.assign(is_oa=df["is_oa"].astype(int), has_abstract=df["has_abstract"].astype(int))[
    ["cites", "citations_per_year", "fwci", "n_authors", "n_institutions", "n_countries", "ref_count", "age_years", "is_oa"]]
corr = num.corr(method="spearman").round(2)
names = {"cites": "Trích dẫn", "citations_per_year": "Trích dẫn/năm", "fwci": "FWCI", "n_authors": "Số tác giả",
         "n_institutions": "Số trường", "n_countries": "Số quốc gia", "ref_count": "Số tài liệu TK", "age_years": "Tuổi bài", "is_oa": "Truy cập mở"}
corr_vi = corr.rename(index=names, columns=names)
fig8 = px.imshow(corr_vi, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="RdBu_r", aspect="auto")
fig8.update_traces(hovertemplate="%{y} × %{x}: ρ = %{z:.2f}<extra></extra>")
fig8.update_layout(coloraxis_colorbar=dict(title="ρ Spearman"))
save(fig8, 8, "quan-he",
     f"Bài càng cũ và càng nhiều tài liệu tham khảo thì càng được trích dẫn (ρ = {VN(corr.loc['age_years', 'cites'], 2)} và {VN(corr.loc['ref_count', 'cites'], 2)}); số tác giả chỉ liên quan yếu (ρ = {VN(corr.loc['n_authors', 'cites'], 2)})",
     "Mỗi ô là hệ số tương quan hạng Spearman giữa hai biến, từ −1 (xanh đậm: ngược chiều) đến +1 (đỏ đậm: cùng chiều). Dùng Spearman vì trích dẫn lệch mạnh và có ngoại lai. Tương quan không có nghĩa là nhân quả.",
     corr.reset_index().rename(columns={"index": "bien"}), "spearman")

### Địa lý & hợp tác

In [26]:
cmap = country_year.dropna(subset=["iso3"]).sort_values("year").copy()
cmap["log_count"] = np.log10(cmap["count"])
fig9 = px.choropleth(cmap, locations="iso3", color="log_count", animation_frame="year", hover_name="name",
                     hover_data={"count": ":,", "log_count": False, "iso3": False, "year": False},
                     range_color=[0, np.ceil(cmap["log_count"].max())], color_continuous_scale="Viridis",
                     labels={"count": "Số bài"}, projection="natural earth")
fig9.update_layout(coloraxis_colorbar=dict(title="Số bài<br>(thang log)", tickvals=[0, 1, 2, 3], ticktext=["1", "10", "100", "1.000"]),
                   height=600, geo=dict(domain=dict(x=[0, 0.9], y=[0.1, 1]), showframe=False, showcoastlines=False))
top_cty = country_year[country_year.year == 2025].nlargest(3, "count")
top_cty = top_cty.assign(name=top_cty["country"].map(VI_NAME).fillna(top_cty["name"]))
vn = country_year[country_year.country == "VN"].set_index("year")["count"]
vn_rank = country_year[country_year.year == 2025].sort_values("count", ascending=False).reset_index(drop=True)
VN_RANK_2025 = int(vn_rank.index[vn_rank.country == "VN"][0]) + 1
save(fig9, 9, "dia-ly",
     f"Năm 2025 dẫn đầu là {', '.join(top_cty['name'])}; Việt Nam đứng thứ {VN_RANK_2025} với {vn.get(2025, 0)} bài",
     "Màu càng sáng thì quốc gia càng có nhiều bài trong năm (thang log, một bài nhiều nước được tính cho mỗi nước). Bấm ▶ để chạy qua các năm hoặc kéo thanh năm; rê chuột để xem số bài.",
     cmap.drop(columns="log_count"), "ban_do")

In [27]:
TOP_N_COUNTRIES, MIN_COAUTH = 30, 10  # N = 10: khoảng 70 cạnh, chỉ 2 nước bị cô lập; N = 15 để lại 7 nước cô lập, N = 5 quá rối (135 cạnh)
paper_cnt = work_country.groupby("country")["id"].nunique()
top30 = set(paper_cnt.nlargest(TOP_N_COUNTRIES).index)
edges = Counter()
for cs in df.loc[df["is_international"], "countries"]:
    cs = [c for c in cs if c in top30]
    edges.update((a, b) for i, a in enumerate(cs) for b in cs[i + 1:])
G = nx.Graph()
G.add_nodes_from(top30)
G.add_weighted_edges_from((a, b, w) for (a, b), w in edges.items() if w >= MIN_COAUTH)
pos = nx.spring_layout(G, weight="weight", seed=7, k=0.9)
names_cc = {c: VI_NAME.get(c) or pycountry.countries.get(alpha_2=c).name for c in top30}
wmax = max(d["weight"] for *_, d in G.edges(data=True))
fig10 = go.Figure()
for a, b, d in G.edges(data=True):
    fig10.add_scatter(x=[pos[a][0], pos[b][0]], y=[pos[a][1], pos[b][1]], mode="lines", hoverinfo="skip", showlegend=False,
                      line=dict(width=0.5 + 6 * d["weight"] / wmax, color="rgba(120,120,120,0.45)"))
deg = dict(G.degree(weight="weight"))
pc = paper_cnt[list(top30)]
size = 12 + 38 * (pc - pc.min()) / (pc.max() - pc.min())
for cont in COLOR_CONT:
    nodes = [c for c in G.nodes if CONT_OF.get(c, "Khác") == cont]
    if not nodes:
        continue
    fig10.add_scatter(x=[pos[c][0] for c in nodes], y=[pos[c][1] for c in nodes], mode="markers+text", name=cont,
                      text=nodes, textposition="middle center", textfont=dict(size=10, color="white"),
                      marker=dict(size=[size[c] for c in nodes], color=COLOR_CONT[cont], line=dict(width=1, color="white")),
                      customdata=[[names_cc[c], pc[c], deg[c], G.degree(c)] for c in nodes],
                      hovertemplate="<b>%{customdata[0]}</b><br>%{customdata[1]:,} bài<br>%{customdata[3]} đối tác (≥ "
                                    + str(MIN_COAUTH) + " bài chung)<extra></extra>")
fig10.update_xaxes(visible=False); fig10.update_yaxes(visible=False)
fig10.update_layout(height=600, legend=dict(title="Châu lục"))
hub = max(G.degree(weight="weight"), key=lambda x: x[1])
hub_partners = G.degree(hub[0])
intl_share = 100 * df.loc[df.n_countries > 0, "is_international"].mean()
top_edge = max(G.edges(data=True), key=lambda e: e[2]["weight"])
edge_table = pd.DataFrame([(a, b, d["weight"]) for a, b, d in G.edges(data=True)], columns=["a", "b", "so_bai_chung"])
save(fig10, 10, "dia-ly",
     f"Chỉ {intl_share:.0f}% số bài có hợp tác quốc tế; {names_cc[hub[0]]} là trung tâm với {hub_partners} đối tác thường xuyên",
     f"Mỗi nút là một trong {TOP_N_COUNTRIES} quốc gia nhiều bài nhất (nút càng to càng nhiều bài, màu theo châu lục). Hai nước được nối khi có ít nhất {MIN_COAUTH} bài chung; nét càng đậm càng nhiều bài chung. Cặp mạnh nhất: {names_cc[top_edge[0]]}–{names_cc[top_edge[1]]} ({top_edge[2]['weight']} bài).",
     edge_table, "network")

### Chủ đề
Kế hoạch ban đầu vẽ sunburst `domain → field → subfield → topic`, nhưng truy vấn đã lọc theo `primary_topic = T14414`, nên mọi bài có cùng một nhánh (Physical Sciences → Computer Science → Information Systems → AI in Education) và sunburst đó chỉ có một vòng. Thay vào đó, sunburst dùng **từ khoá** OpenAlex (99,9% bài có), gom thành 6 nhóm chủ đề bằng quy tắc từ khoá, tô màu theo tỷ lệ bài từ 2023 trở đi. OpenAlex gán từ khoá bằng tiếng Anh cho mọi bài, nên không cần lọc theo ngôn ngữ.

In [28]:
GENERIC = {"artificial intelligence in education", "higher education", "artificial intelligence in higher education",
           "artificial intelligence", "university students", "higher education institutions", "undergraduate students",
           "graduate students", "university teaching", "university faculty", "research trends"}
THEMES = [
    ("AI tạo sinh & LLM", ["chatgpt", "generative", "large language model", "llm", "gpt", "chatbot", "prompt", "ai-generated"]),
    ("Đạo đức, liêm chính & quản trị", ["ethic", "integrity", "plagiarism", "privacy", "bias", "governance", "policy", "responsible",
                                         "trust", "overreliance", "dependency", "cheating", "equity", "divide"]),
    ("Cá nhân hoá & hệ thống thông minh", ["personalized", "adaptive", "tutoring", "analytics", "intelligent", "machine learning", "predict"]),
    ("Năng lực & kỹ năng", ["literacy", "critical thinking", "skill", "creativity", "competenc", "self-regulat", "autonomy", "self-directed"]),
    ("Chấp nhận & thái độ", ["acceptance", "perception", "attitude", "adoption", "perceived", "intention", "utaut", "motivation", "engagement"]),
    ("Giảng dạy, đánh giá & chương trình", ["teaching", "assessment", "curriculum", "feedback", "writing", "teacher", "pedagog", "learning outcome",
                                             "academic performance", "language learning", "online learning", "educational technology"]),
]
def theme_of(k):
    kl = k.lower()
    return next((t for t, keys in THEMES if any(s in kl for s in keys)), None)

kw = work_kw[~work_kw["keyword"].str.lower().isin(GENERIC)].copy()
kw["theme"] = kw["keyword"].map(theme_of)
kw["post"] = kw["year"] >= 2023
kw["recent"] = kw["year"] >= 2025
RECENT_AVG = 100 * (df["year"] >= 2025).mean()
kw_stats = (kw.dropna(subset=["theme"]).groupby(["theme", "keyword"])
            .agg(so_bai=("id", "nunique"), ty_le_2025_2026=("recent", "mean")).reset_index())
kw_top = kw_stats.sort_values("so_bai", ascending=False).groupby("theme").head(6).copy()
kw_top["ty_le_2025_2026"] = (100 * kw_top["ty_le_2025_2026"]).round(1)
theme_papers = kw.dropna(subset=["theme"]).groupby("theme")["id"].nunique().sort_values(ascending=False)
fig11 = px.sunburst(kw_top, path=["theme", "keyword"], values="so_bai", color="ty_le_2025_2026",
                    color_continuous_scale="RdBu_r", color_continuous_midpoint=RECENT_AVG, range_color=[RECENT_AVG - 25, RECENT_AVG + 25],
                    labels={"so_bai": "Số bài", "ty_le_2025_2026": "% bài 2025–2026"})
fig11.update_traces(hovertemplate="<b>%{label}</b><br>%{value:,} bài<br>%{color:.0f}% bài thuộc 2025–2026<extra></extra>", insidetextorientation="radial")
fig11.update_layout(height=640, coloraxis_colorbar=dict(title="% bài<br>2025–2026", ticksuffix="%"))
big_kw = kw_top[kw_top["so_bai"] >= 1000]  # chỉ so các từ khoá lớn để kết luận không dựa vào nhóm nhỏ
rising = big_kw.sort_values("ty_le_2025_2026", ascending=False).iloc[0]
fading = big_kw.sort_values("ty_le_2025_2026").iloc[0]
genai_share = 100 * theme_papers["AI tạo sinh & LLM"] / len(df)
theme_post = 100 * kw.dropna(subset=["theme"]).drop_duplicates(["id", "theme"]).groupby("theme")["post"].mean()
big2 = theme_papers.index[:2].tolist()
save(fig11, 11, "chu-de",
     f"Hai nhóm lớn nhất là '{big2[0]}' ({100 * theme_papers[big2[0]] / len(df):.0f}% số bài) và '{big2[1]}' ({100 * theme_papers[big2[1]] / len(df):.0f}%); "
     f"trong các từ khoá trên 1.000 bài, '{rising.keyword}' đang lên nhanh nhất, '{fading.keyword}' đang nhạt dần",
     f"Vòng trong là 6 nhóm chủ đề gom từ từ khoá OpenAlex, vòng ngoài là 6 từ khoá nhiều bài nhất của mỗi nhóm; độ lớn là số bài. Màu so với mức trung bình {RECENT_AVG:.0f}% bài thuộc 2025–2026: đỏ là từ khoá đang lên (tập trung ở bài mới), xanh là từ khoá đang nhạt dần. Bấm vào một nhóm để phóng to.",
     kw_top, "sunburst_chu_de")

In [29]:
PERIODS = [("2015–2020", 2015, 2020), ("2021–2022", 2021, 2022), ("2023–2024", 2023, 2024), ("2025–2026", 2025, 2026)]
kw["p4"] = pd.cut(kw["year"], bins=[2014, 2020, 2022, 2024, 2026], labels=[p[0] for p in PERIODS])
papers_p4 = pd.cut(df["year"], bins=[2014, 2020, 2022, 2024, 2026], labels=[p[0] for p in PERIODS]).value_counts()
kwp = kw.groupby(["p4", "keyword"], observed=True)["id"].nunique().rename("so_bai").reset_index()
kwp["ty_le"] = 100 * kwp["so_bai"] / kwp["p4"].map(papers_p4).astype(float)
chosen = set(kwp.sort_values("ty_le", ascending=False).groupby("p4", observed=True).head(10)["keyword"])
grid = pd.MultiIndex.from_product([[p[0] for p in PERIODS], sorted(chosen)], names=["p4", "keyword"])
kw12 = kwp.set_index(["p4", "keyword"]).reindex(grid, fill_value=0).reset_index()
last = kw12[kw12.p4 == PERIODS[-1][0]].set_index("keyword")["ty_le"].sort_values()
kw12["keyword"] = pd.Categorical(kw12["keyword"], categories=last.index, ordered=True)
kw12 = kw12.sort_values(["p4", "keyword"])
fig12 = px.bar(kw12, x="ty_le", y="keyword", orientation="h", animation_frame="p4", range_x=[0, kw12["ty_le"].max() * 1.1],
               color_discrete_sequence=["#0072B2"], labels={"ty_le": "% số bài trong giai đoạn", "keyword": "", "p4": "Giai đoạn"},
               hover_data={"so_bai": ":,", "ty_le": ":.1f"})
fig12.update_xaxes(ticksuffix="%")
fig12.update_layout(height=160 + 28 * len(chosen), yaxis=dict(categoryorder="array", categoryarray=list(last.index)))
first_p, last_p = PERIODS[0][0], PERIODS[-1][0]
kv = kw12.set_index(["p4", "keyword"])["ty_le"]
TOP_FIRST = kv[first_p].sort_values(ascending=False).index[:2].tolist()
TOP_LAST = kv[last_p].sort_values(ascending=False).index[:2].tolist()
save(fig12, 12, "chu-de",
     f"Từ khoá dịch từ '{TOP_FIRST[0]}', '{TOP_FIRST[1]}' ({first_p}) sang '{TOP_LAST[0]}', '{TOP_LAST[1]}': '{TOP_LAST[0]}' có mặt trong {kv[(last_p, TOP_LAST[0])]:.0f}% bài {last_p}",
     f"Mỗi thanh là phần trăm số bài của giai đoạn có từ khoá đó (bỏ các từ chung như 'higher education'). Bấm ▶ hoặc kéo thanh giai đoạn để xem từ khoá thay đổi; thứ tự xếp theo giai đoạn {last_p}.",
     kw12, "tu_khoa_giai_doan")

### Dạy và học AI tại Việt Nam so với thế giới
Toàn bộ dữ liệu đã thuộc chủ đề *AI in Education* ở bậc đại học, nên phần này không lọc lại chủ đề. Thay vào đó, phần này tách **các bài có ít nhất một tác giả ở Việt Nam** và so với thế giới cùng 5 nước tham chiếu: Thái Lan, Malaysia, Indonesia (ASEAN), Trung Quốc và Hoa Kỳ (hai nước nhiều bài nhất). Để quy mô các nước so được với nhau, mọi so sánh dùng số đã chuẩn hoá: ‰ số bài thế giới, % số bài của nước, FWCI.

Thêm một bộ lọc con, gọi là **"học về AI"**: bài coi AI là *nội dung cần dạy và học* (AI literacy, năng lực AI, chương trình AI, kỹ năng AI, tư duy tính toán), phân biệt với bài dùng AI làm *công cụ* hỗ trợ dạy học. Bài được gắn nhãn khi có từ khoá OpenAlex khớp, hoặc tiêu đề chứa cụm như "AI literacy", "AI curriculum", "teaching AI".


In [30]:
from statsmodels.stats.proportion import proportions_ztest

# Bộ lọc "học về AI": AI là nội dung cần dạy/học (năng lực, chương trình, kỹ năng), không chỉ là công cụ hỗ trợ dạy học.
ABOUT_AI_KW = (r"\bai literac|\bai competenc|\bai skills|\bai readiness|\bai curricul|ai integration in curriculum"
               r"|machine learning education|data science (?:education|curricul)|computational thinking")
ABOUT_AI_TITLE = (r"\b(?:ai|artificial intelligence|machine learning|data science)\s+(?:literacy|competenc\w*|curricul\w*|courses?|skills?|readiness)\b"
                  r"|\bteaching (?:ai|artificial intelligence|machine learning)\b")
about_kw = set(work_kw.loc[work_kw["keyword"].str.lower().str.contains(ABOUT_AI_KW), "id"])
about_title = set(df.loc[df["title"].str.lower().str.contains(ABOUT_AI_TITLE), "id"])
df["about_ai"] = df["id"].isin(about_kw | about_title)
# Kiểm tra độ nhất quán: bài được gắn nhãn có nhắc trực tiếp các cụm này trong tiêu đề hoặc tóm tắt không?
EXPLICIT = r"literac|competenc|curricul|course|skill|readiness|computational thinking|teaching (?:ai|artificial intelligence|machine learning)"
vn_text = (df["title"].fillna("") + " " + df["abstract"].fillna("")).str.lower()
explicit_rate = 100 * vn_text[df["about_ai"]].str.contains(EXPLICIT).mean()
explicit_rest = 100 * vn_text[~df["about_ai"]].str.contains(EXPLICIT).mean()

REF = ["VN", "TH", "MY", "ID", "CN", "US"]  # Việt Nam, 3 nước ASEAN, 2 nước nhiều bài nhất
REF_NAME = {**{cc: VI_NAME[cc] for cc in REF if cc in VI_NAME}, "TH": "Thái Lan", "WORLD": "Thế giới"}
vn_ids = set(work_country.loc[work_country["country"] == "VN", "id"])
df["is_vn"] = df["id"].isin(vn_ids)
vn_df = df[df["is_vn"]]
prec_vn = prec.assign(id=prec["id"].str.split("/").str[-1])
prec_vn = prec_vn[(prec_vn["bien_the"] == "F") & prec_vn["id"].isin(vn_ids)]["đúng"]
print(f"Bài có tác giả Việt Nam: {len(vn_df)} / {len(df)} ({1000 * len(vn_df) / len(df):.1f}‰)")
print(f"Precision: chỉ {len(prec_vn)} bài Việt Nam rơi vào mẫu 100 bài ({prec_vn.sum()}/{len(prec_vn)} đúng) → không đủ để ước lượng riêng")
print(f"'Học về AI': {df['about_ai'].sum():,} bài ({100 * df['about_ai'].mean():.1f}%) | "
      f"{explicit_rate:.0f}% số bài này nhắc trực tiếp các cụm trong tiêu đề/tóm tắt (nhóm còn lại: {explicit_rest:.0f}%)")
print("Ngôn ngữ của bài Việt Nam:", vn_df["language"].fillna("thiếu nhãn").value_counts().to_dict())
display(vn_df[vn_df["language"].isin(["vi", "ang"])][["language", "title"]])


Bài có tác giả Việt Nam: 285 / 23540 (12.1‰)
Precision: chỉ 3 bài Việt Nam rơi vào mẫu 100 bài (3/3 đúng) → không đủ để ước lượng riêng
'Học về AI': 3,265 bài (13.9%) | 94% số bài này nhắc trực tiếp các cụm trong tiêu đề/tóm tắt (nhóm còn lại: 46%)
Ngôn ngữ của bài Việt Nam: {'en': 178, 'thiếu nhãn': 96, 'vi': 5, 'ang': 4, 'ru': 1, 'ca': 1}


,language,title
12692,ang,Using Generative AI Tool ChatGPT in Language Teaching from University Language Teacher...
13160,vi,EXPLORING PREDICTORS OF AI ADOPTION IN IMPROVING LEGAL TEXT COMPREHENSION: AN EMPIRICA...
13751,vi,TIẾP CẬN AI-NATIVE TRONG ĐẠI HỌC: QUAN SÁT TỪ HỢP TÁC VỚI OPENAI
14719,vi,TIẾP CẬN AI-NATIVE TRONG ĐẠI HỌC: TÍCH HỢP THỂ CHẾ VÀ BẢN ĐỊA HÓA NHẬN THỨC QUA HAI HỒ...
16640,vi,Teachers’ and students’ perceptions of AI-based writing assessment
16653,vi,USING CHATGPT AS A WRITING ASSISTANT: BENEFITS AND CHALLENGES FOR EFL STUDENTS AT DONG...
18671,ang,Teacher Well-Being in the Age of Artificial Intelligence: Emerging Challenges and Peda...
19200,ang,A Qualitative Case Study of Pre-Service Teachers' Perceptions of Using ChatGPT for Ped...
20283,ang,Utilizing Google NotebookLM in Teaching and Research at Ho Chi Minh City University of...


In [31]:
world_n = yearly.set_index("year")["count"]
cy_ref = (country_year[country_year["country"].isin(REF)].pivot(index="year", columns="country", values="count")
          .reindex(index=range(2022, 2027), columns=REF).fillna(0))  # từ 2022: năm cuối trước ChatGPT; trước đó mỗi nước chỉ vài bài
per1000 = (1000 * cy_ref.div(world_n.reindex(cy_ref.index), axis=0)).round(2)
t13 = (per1000.reset_index().melt(id_vars="year", var_name="country", value_name="tren_1000_bai")
       .merge(cy_ref.reset_index().melt(id_vars="year", var_name="country", value_name="so_bai"), on=["year", "country"]))
t13["ten"] = t13["country"].map(REF_NAME)
COLOR_REF = {"Việt Nam": "#D55E00", "Thái Lan": "#56B4E9", "Malaysia": "#009E73", "Indonesia": "#CC79A7",
             "Trung Quốc": "#0072B2", "Hoa Kỳ": "#999999", "Thế giới": "#1d2433"}
fig13 = px.line(t13, x="year", y="tren_1000_bai", color="ten", markers=True, color_discrete_map=COLOR_REF,
                category_orders={"ten": [REF_NAME[c] for c in REF]}, custom_data=["so_bai"],
                labels={"year": "Năm", "tren_1000_bai": "Số bài trên 1.000 bài của thế giới", "ten": "Quốc gia"})
fig13.update_traces(hovertemplate="%{fullData.name} %{x}: %{y:.1f}‰ (%{customdata[0]:,} bài)<extra></extra>")
fig13.update_traces(selector=dict(name="Việt Nam"), line=dict(width=4))
fig13.update_xaxes(dtick=1)
fig13.update_yaxes(ticksuffix="‰", rangemode="tozero")
CAGR_REF = {cc: (cy_ref.loc[2025, cc] / cy_ref.loc[2023, cc]) ** 0.5 - 1 for cc in REF}
CAGR_REF["WORLD"] = (world_n[2025] / world_n[2023]) ** 0.5 - 1
rank_by_year = {y: g.sort_values("count", ascending=False)["country"].tolist() for y, g in country_year.groupby("year")}
VN_RANK = {y: r.index("VN") + 1 for y, r in rank_by_year.items() if "VN" in r}
save(fig13, 13, "viet-nam",
     f"Việt Nam tăng từ {VN(per1000.loc[2023, 'VN'])}‰ (2023) lên {VN(per1000.loc[2026, 'VN'])}‰ số bài thế giới (2026), "
     f"gần Thái Lan ({VN(per1000.loc[2026, 'TH'])}‰), trong khi Indonesia là {VN(per1000.loc[2026, 'ID'])}‰",
     "Mỗi đường là số bài có tác giả của một nước trên mỗi 1.000 bài của cả thế giới trong năm (chuẩn hoá theo quy mô chung, nên năm 2026 chưa đủ năm vẫn so được). "
     "Bắt đầu từ 2022 vì các năm trước mỗi nước chỉ có vài bài. Số đếm lấy từ group_by của OpenAlex trên toàn bộ dữ liệu.",
     t13, "vn_ty_trong")
pd.DataFrame({"CAGR 2023–2025 (%)": {REF_NAME[cc]: round(100 * v) for cc, v in CAGR_REF.items()},
              "‰ 2026": {REF_NAME[cc]: per1000.loc[2026, cc] for cc in REF}})


,CAGR 2023–2025 (%),‰ 2026
Việt Nam,220,13.76
Thái Lan,162,12.48
Malaysia,185,35.09
Indonesia,209,75.97
Trung Quốc,206,85.60
Hoa Kỳ,142,67.13
Thế giới,172,NaN


In [32]:
from plotly.subplots import make_subplots

rng_vn = np.random.default_rng(2026)  # rng riêng, không làm đổi kết quả bootstrap ở phần 8
def boot_ci(x, n_boot=2000):
    means = np.array([rng_vn.choice(x, len(x)).mean() for _ in range(n_boot)])
    return np.percentile(means, [2.5, 97.5])

groups = {cc: df[df["id"].isin(set(work_country.loc[work_country["country"] == cc, "id"]))] for cc in REF}
groups["WORLD"] = df
rows14 = []
for cc, g in groups.items():
    lo, hi = boot_ci(g["fwci"].to_numpy())
    has_cty = g[g["n_countries"] > 0]
    rows14.append({"country": cc, "ten": REF_NAME[cc], "so_bai": len(g),
                   "hop_tac_quoc_te_%": round(100 * has_cty["is_international"].mean(), 1),
                   "fwci_tb": round(g["fwci"].mean(), 2), "fwci_ci_low": round(lo, 2), "fwci_ci_high": round(hi, 2),
                   "co_trich_dan_%": round(100 * (g["cites"] > 0).mean(), 1),
                   "bai_2025_2026_%": round(100 * (g["year"] >= 2025).mean(), 1),
                   "hoc_ve_ai_%": round(100 * g["about_ai"].mean(), 1),
                   "truy_cap_mo_%": round(100 * g["is_oa"].mean(), 1)})
t14 = pd.DataFrame(rows14)
order14 = t14["ten"].tolist()[::-1]
METRICS14 = [("hop_tac_quoc_te_%", "Hợp tác quốc tế (% bài)"), ("fwci_tb", "FWCI trung bình (KTC 95%)"),
             ("hoc_ve_ai_%", "Bài 'học về AI' (%)"), ("bai_2025_2026_%", "Bài thuộc 2025–2026 (%)")]
fig14 = make_subplots(rows=1, cols=4, shared_yaxes=True, subplot_titles=[m[1] for m in METRICS14], horizontal_spacing=0.04)
for k, (col, label) in enumerate(METRICS14, start=1):
    err = dict(type="data", symmetric=False, array=t14["fwci_ci_high"] - t14["fwci_tb"],
               arrayminus=t14["fwci_tb"] - t14["fwci_ci_low"], color="#555") if col == "fwci_tb" else None
    fig14.add_bar(x=t14[col], y=t14["ten"], orientation="h", error_x=err, showlegend=False, row=1, col=k,
                  marker_color=[COLOR_REF[n] for n in t14["ten"]], customdata=t14[["so_bai"]],
                  text=t14[col].map(lambda v: VN(v, 2 if col == "fwci_tb" else 0)), textposition="inside", insidetextanchor="start", textangle=0, textfont=dict(color="white"),
                  hovertemplate="%{y}: %{x}<br>%{customdata[0]:,} bài<extra>" + label + "</extra>")
fig14.update_yaxes(categoryorder="array", categoryarray=order14)
fig14.update_layout(height=420, annotations=[dict(font=dict(size=12)) for _ in METRICS14])
t14i = t14.set_index("country")
save(fig14, 14, "viet-nam",
     f"Việt Nam hợp tác quốc tế ở {VN(t14i.loc['VN', 'hop_tac_quoc_te_%'], 0)}% số bài, ngang thế giới ({VN(t14i.loc['WORLD', 'hop_tac_quoc_te_%'], 0)}%) "
     f"nhưng chỉ bằng một nửa Thái Lan ({VN(t14i.loc['TH', 'hop_tac_quoc_te_%'], 0)}%); FWCI {VN(t14i.loc['VN', 'fwci_tb'], 2)} chưa khác thế giới ({VN(t14i.loc['WORLD', 'fwci_tb'], 2)}) một cách chắc chắn",
     "Mỗi cột là một chỉ số, mỗi thanh là một nước (đếm mọi bài có ít nhất một tác giả ở nước đó). FWCI là trích dẫn đã chuẩn hoá theo năm xuất bản và lĩnh vực (1 = trung bình thế giới); "
     "vạch đen là khoảng tin cậy bootstrap 95%, hai khoảng chồng lên nhau thì chưa đủ bằng chứng là khác nhau. 'Học về AI' là bài coi AI là nội dung cần dạy (AI literacy, năng lực, chương trình AI).",
     t14, "vn_so_sanh")
t14


,country,ten,so_bai,hop_tac_quoc_te_%,fwci_tb,fwci_ci_low,fwci_ci_high,co_trich_dan_%,bai_2025_2026_%,hoc_ve_ai_%,truy_cap_mo_%
0,VN,Việt Nam,285,13.7,4.21,2.66,6.13,40.4,81.4,17.2,78.9
1,TH,Thái Lan,280,27.5,3.64,2.57,4.93,31.8,80.7,18.9,89.6
2,MY,Malaysia,742,46.6,5.22,3.82,6.77,39.9,82.6,16.3,82.5
3,ID,Indonesia,1682,16.0,1.77,1.37,2.26,26.4,81.8,15.4,91.9
4,CN,Trung Quốc,2217,23.0,6.54,5.71,7.43,45.1,75.9,11.9,69.2
5,US,Hoa Kỳ,1546,25.1,7.75,6.61,8.93,51.2,74.8,17.5,69.0
6,WORLD,Thế giới,23540,15.0,3.93,3.73,4.14,35.5,76.9,13.9,77.3


In [33]:
theme_ids = kw.dropna(subset=["theme"]).groupby("theme")["id"].agg(set)
theme_ids["Học về AI (năng lực, chương trình)"] = set(df.loc[df["about_ai"], "id"])
rest = df[~df["is_vn"]]  # so Việt Nam với phần còn lại để hai mẫu độc lập khi kiểm định
rows15 = []
for theme, ids_t in theme_ids.items():
    k_vn, k_rest = vn_df["id"].isin(ids_t).sum(), rest["id"].isin(ids_t).sum()
    _, pval = proportions_ztest([k_vn, k_rest], [len(vn_df), len(rest)])
    rows15.append({"chu_de": theme, "viet_nam_%": round(100 * k_vn / len(vn_df), 1), "the_gioi_%": round(100 * df["id"].isin(ids_t).mean(), 1),
                   "con_lai_%": round(100 * k_rest / len(rest), 1), "p_value": round(float(pval), 4)})
t15 = pd.DataFrame(rows15)
t15["chenh_diem_%"] = (t15["viet_nam_%"] - t15["con_lai_%"]).round(1)
t15["co_y_nghia"] = t15["p_value"] < 0.05
t15 = t15.sort_values("viet_nam_%").reset_index(drop=True)
fig15 = go.Figure()
for _, row15 in t15.iterrows():
    fig15.add_scatter(x=[row15["the_gioi_%"], row15["viet_nam_%"]], y=[row15["chu_de"]] * 2, mode="lines", showlegend=False,
                      hoverinfo="skip", line=dict(color="#bbb", width=3))
fig15.add_scatter(x=t15["the_gioi_%"], y=t15["chu_de"], mode="markers", name="Thế giới", marker=dict(size=13, color="#1d2433"),
                  hovertemplate="Thế giới: %{x:.1f}% số bài<extra></extra>")
fig15.add_scatter(x=t15["viet_nam_%"], y=t15["chu_de"], mode="markers+text", name="Việt Nam", marker=dict(size=13, color="#D55E00"),
                  text=np.where(t15["co_y_nghia"], "  *", ""), textposition="middle right", customdata=t15[["p_value"]],
                  hovertemplate="Việt Nam: %{x:.1f}% số bài<br>p = %{customdata[0]:.3f}<extra></extra>")
fig15.update_xaxes(title="% số bài có chủ đề", ticksuffix="%", rangemode="tozero")
fig15.update_layout(height=430, legend=dict(orientation="h", y=1.08, x=0))
sig = t15[t15["co_y_nghia"]].sort_values("chenh_diem_%")
big_gap = t15.loc[t15["chenh_diem_%"].abs().idxmax()]
save(fig15, 15, "viet-nam",
     f"Nghiên cứu của Việt Nam nghiêng về '{big_gap.chu_de}' ({VN(big_gap['viet_nam_%'], 0)}% so với {VN(big_gap['the_gioi_%'], 0)}% thế giới); "
     f"{len(sig)}/{len(t15)} khác biệt có ý nghĩa thống kê",
     "Mỗi hàng là một nhóm chủ đề (gom từ từ khoá OpenAlex như biểu đồ ⑪, thêm nhóm 'Học về AI'). Chấm đen là tỷ lệ bài của cả thế giới, chấm cam là của Việt Nam; "
     "một bài có thể thuộc nhiều chủ đề. Dấu * đánh dấu khác biệt có ý nghĩa (kiểm định z hai tỷ lệ, Việt Nam so với phần còn lại, p < 0,05). Với 285 bài, chênh dưới khoảng 6 điểm % thường chưa đủ chắc chắn.",
     t15, "vn_chu_de")

# Từ khoá đặc trưng: xuất hiện ở ≥ 10 bài Việt Nam, tỷ lệ ở Việt Nam cao hơn thế giới nhiều nhất (bỏ tên nước)
kw_vn = work_kw[work_kw["id"].isin(vn_ids)].groupby("keyword")["id"].nunique()
kw_all = work_kw.groupby("keyword")["id"].nunique()
lift = pd.DataFrame({"so_bai_vn": kw_vn, "vn_%": 100 * kw_vn / len(vn_df), "the_gioi_%": 100 * kw_all[kw_vn.index] / len(df)})
lift["lift"] = lift["vn_%"] / lift["the_gioi_%"]
lift = lift[(lift["so_bai_vn"] >= 10) & ~lift.index.str.contains("Viet", case=False)].sort_values("lift", ascending=False).round(2)
lift.head(8)


,so_bai_vn,vn_%,the_gioi_%,lift
keyword,,,,
English majors,18,6.32,0.23,27.53
English language learning,10,3.51,0.78,4.49
artificial intelligence in language teaching,14,4.91,1.33,3.68
institutional support,11,3.86,1.05,3.68
writing skills,10,3.51,0.99,3.54
AI-assisted language learning,16,5.61,1.59,3.53
English as a foreign language,13,4.56,1.30,3.51
perceived ease of use,13,4.56,1.52,3.01


In [34]:
vn_paper_inst, vn_inst_name = {}, {}
for w in works_raw:
    wid = w["id"].split("/")[-1]
    if wid in vn_ids:
        found = {(i["id"].split("/")[-1], i["display_name"]) for a in w["authorships"]
                 for i in a.get("institutions", []) if i.get("country_code") == "VN"}
        vn_paper_inst[wid] = {iid for iid, _ in found}
        vn_inst_name.update(found)
t16 = pd.DataFrame(Counter(i for s in vn_paper_inst.values() for i in s).most_common(), columns=["inst", "so_bai"])
t16["ten"] = t16["inst"].map(vn_inst_name)
t16["ty_le_bai_vn_%"] = (100 * t16["so_bai"] / len(vn_df)).round(1)
top3 = set(t16["inst"][:3])
VN_TOP3_SHARE = 100 * np.mean([bool(s & top3) for s in vn_paper_inst.values()])
top16 = t16.head(10).iloc[::-1]
fig16 = px.bar(top16, x="so_bai", y="ten", orientation="h", text="so_bai", color_discrete_sequence=["#D55E00"],
               custom_data=["ty_le_bai_vn_%"], labels={"so_bai": "Số bài có tác giả của trường", "ten": ""})
fig16.update_traces(textposition="outside", hovertemplate="%{y}: %{x} bài (%{customdata[0]:.0f}% số bài Việt Nam)<extra></extra>")
fig16.update_layout(height=430)
save(fig16, 16, "viet-nam",
     f"{len(t16)} trường/viện Việt Nam có bài; dẫn đầu là {t16['ten'][0]} ({t16['so_bai'][0]} bài) và {t16['ten'][1]} ({t16['so_bai'][1]} bài)",
     f"Mỗi thanh là số bài có ít nhất một tác giả thuộc trường, đếm theo ID cơ sở của OpenAlex, nên trường thành viên (ví dụ VNU University of Education) "
     f"được tính riêng với ĐHQG Hà Nội. Ba trường đầu cùng góp mặt trong {VN_TOP3_SHARE:.0f}% số bài của Việt Nam. Rê chuột để xem tỷ lệ.",
     t16, "vn_truong")


In [35]:
vn_partners = Counter(c2 for cs in vn_df["countries"] for c2 in cs if c2 != "VN")
lang_vn = vn_df["language"].fillna("thiếu nhãn").value_counts()
VN_INSIGHT = {
    "n": len(vn_df), "per1000_all": round(1000 * len(vn_df) / len(df), 1),
    "by_year": country_year[country_year["country"] == "VN"].set_index("year")["count"].to_dict(),
    "per1000": {cc: {int(y): float(v) for y, v in per1000[cc].items()} for cc in REF},
    "rank": {int(y): v for y, v in VN_RANK.items() if y >= 2023},
    "cagr_2023_2025": {cc: round(100 * v) for cc, v in CAGR_REF.items()},
    "compare": t14.set_index("country").drop(columns="ten").to_dict("index"),
    "names": REF_NAME,
    "themes": t15.set_index("chu_de")[["viet_nam_%", "the_gioi_%", "p_value", "co_y_nghia"]].to_dict("index"),
    "lift_top": [[k, int(r.so_bai_vn), float(r["vn_%"]), float(r["the_gioi_%"])] for k, r in lift.head(5).iterrows()],
    "partners": vn_partners.most_common(5), "n_intl": int(vn_df["is_international"].sum()),
    "n_inst": len(t16), "top_inst": t16[["ten", "so_bai"]].head(5).values.tolist(), "top3_share": round(VN_TOP3_SHARE, 1),
    "about_ai_world": round(100 * df["about_ai"].mean(), 1), "about_ai_n": int(df["about_ai"].sum()),
    "about_ai_explicit": round(explicit_rate, 1), "about_ai_explicit_rest": round(explicit_rest, 1),
    "prec_k": len(prec_vn), "prec_ok": int(prec_vn.sum()),
    "lang": {k: int(v) for k, v in lang_vn.items()},
    "type": {k: int(v) for k, v in vn_df["type"].value_counts().items()},
}
display(t15)
{k: VN_INSIGHT[k] for k in ["n", "by_year", "rank", "partners", "n_intl", "top_inst", "top3_share"]}


,chu_de,viet_nam_%,the_gioi_%,con_lai_%,p_value,chenh_diem_%,co_y_nghia
0,"Học về AI (năng lực, chương trình)",17.2,13.9,13.8,0.1025,3.4,False
1,Cá nhân hoá & hệ thống thông minh,19.6,21.8,21.8,0.3769,-2.2,False
2,Năng lực & kỹ năng,38.6,39.0,39.0,0.8878,-0.4,False
3,"Đạo đức, liêm chính & quản trị",47.0,46.2,46.2,0.7901,0.8,False
4,Chấp nhận & thái độ,54.4,42.4,42.2,0.0000,12.2,True
5,AI tạo sinh & LLM,64.9,53.4,53.3,0.0001,11.6,True
6,"Giảng dạy, đánh giá & chương trình",68.1,60.5,60.4,0.0085,7.7,True


{'n': 285,
 'by_year': {2018: 2,
  2020: 1,
  2022: 1,
  2023: 9,
  2024: 40,
  2025: 92,
  2026: 140},
 'rank': {2023: 34, 2024: 25, 2025: 21, 2026: 17},
 'partners': [('AU', 13), ('GB', 8), ('SG', 4), ('US', 4), ('IN', 3)],
 'n_intl': 39,
 'top_inst': [['FPT University', 34],
  ['Vietnam National University, Hanoi', 28],
  ['VNU University of Education', 14],
  ['British University Vietnam', 13],
  ['University of Languages and International Studies', 13]],
 'top3_share': np.float64(22.8)}

## 8. Insight
### Bootstrap: trích dẫn trước và sau 2023 khác nhau thật hay do ngẫu nhiên?
Hơn một nửa số bài từ 2023 chưa có trích dẫn nên trung vị của cả hai giai đoạn gần như bằng 0 và không phân biệt được gì. Vì vậy so **trung bình**: lấy mẫu lại có hoàn lại 2.000 lần trong mỗi giai đoạn, tính chênh lệch trung bình và khoảng tin cậy 95% (phân vị 2,5 và 97,5). Bootstrap không giả định phân phối chuẩn nên dùng được cho dữ liệu lệch như trích dẫn. So hai thước đo: `citations_per_year` (bị độ trễ trích dẫn ảnh hưởng) và FWCI (OpenAlex đã chuẩn hoá theo năm xuất bản và lĩnh vực; 1 = trung bình thế giới).

In [36]:
rng = np.random.default_rng(42)
def boot_mean_diff(a, b, n_boot=2000):
    diffs = np.array([rng.choice(b, len(b)).mean() - rng.choice(a, len(a)).mean() for _ in range(n_boot)])
    return b.mean() - a.mean(), *np.percentile(diffs, [2.5, 97.5])

pre, post = df[df.period == "2015–2022"], df[df.period == "2023–2026"]
BOOT = {}
for col in ["citations_per_year", "fwci"]:
    d, lo, hi = boot_mean_diff(pre[col].to_numpy(), post[col].to_numpy())
    BOOT[col] = {"truoc": float(pre[col].mean()), "sau": float(post[col].mean()), "chenh": float(d), "ci_low": float(lo), "ci_high": float(hi)}
pd.DataFrame(BOOT).T.round(3)

,truoc,sau,chenh,ci_low,ci_high
citations_per_year,7.559,2.895,-4.664,-8.606,-2.139
fwci,3.770,3.938,0.168,-0.707,1.027


Khoảng tin cậy không chứa 0 thì khác biệt không phải do ngẫu nhiên lấy mẫu. Nhưng với `citations_per_year`, khác biệt một phần là do bài mới chưa kịp được trích dẫn. FWCI loại bớt ảnh hưởng đó, nên là thước đo công bằng hơn khi so hai giai đoạn. Phần diễn giải nằm ở báo cáo insight.

In [37]:
INSIGHTS = {
    "download_date": str(DOWNLOAD_DATE.date()), "filter": LOG["filter"], "api_count": API_COUNT, "n_clean": len(df),
    "n_countries": int(country_year["country"].nunique()), "n_institutions": len(insts), "n_edges": G.number_of_edges(), "min_coauth": MIN_COAUTH,
    "precision": round(PRECISION), "precision_margin": round(MARGIN, 1), "precision_k": K,
    "count_2022": int(c[2022]), "count_2025": int(c[2025]), "count_2026": int(c[2026]), "mult_2025_2022": round(float(mult), 1),
    "share_2023_on": round(100 * float(c[c.index >= 2023].sum()) / API_COUNT, 1),
    "cagr_2018_2022": round(100 * CAGR_PRE), "cagr_2022_2025": round(100 * CAGR_POST), "yoy_std": round(YOY_STD),
    "yoy": {int(k): float(v) for k, v in zip(y4.year, y4["yoy_%"])},
    "rolling_2022_12": round(float(r["2022-12-01"])), "rolling_last": round(float(r[LAST_FULL_MONTH])), "last_month": f"{LAST_FULL_MONTH:%m/%Y}",
    "asia_2019": round(float(s[(2019, "Châu Á")]), 1), "asia_2026": round(float(s[(2026, "Châu Á")]), 1),
    "europe_2019": round(float(s[(2019, "Châu Âu")]), 1), "europe_2026": round(float(s[(2026, "Châu Âu")]), 1),
    "na_2019": round(float(s[(2019, "Bắc Mỹ")]), 1), "na_2026": round(float(s[(2026, "Bắc Mỹ")]), 1),
    "top3_2025": top_cty[["name", "count"]].values.tolist(), "vn_2025": int(vn.get(2025, 0)), "vn_2022": int(vn.get(2022, 0)),
    "vn_rank_2025": VN_RANK_2025, "cited_pre": round(float(cited["2015–2022"]), 1), "cited_post": round(float(cited["2023–2026"]), 1),
    "top_first": TOP_FIRST, "top_last": TOP_LAST, "top_last_share": round(float(kv[(last_p, TOP_LAST[0])]), 1), "vn_total": int(vn.sum()),
    "intl_share": round(intl_share, 1), "hub": names_cc[hub[0]], "hub_degree": hub_partners,
    "genai_share": round(genai_share, 1), "genai_post": round(float(theme_post["AI tạo sinh & LLM"]), 1), "rising_kw": rising.keyword, "rising_pct": float(rising.ty_le_2025_2026),
    "fading_kw": fading.keyword, "fading_pct": float(fading.ty_le_2025_2026), "recent_avg": round(RECENT_AVG, 1), "theme_papers": {k: int(v) for k, v in theme_papers.items()},
    "chatgpt_2324": round(float(kv[("2023–2024", "ChatGPT")]), 1), "chatgpt_last": round(float(kv[(last_p, "ChatGPT")]), 1),
    "integrity_last": round(float(kv.get((last_p, "academic integrity"), 0)), 1),
    "slope": round(float(SLOPE), 2), "r2": round(float(R2), 2), "zero_cites": round(zero_share, 1),
    "boot": BOOT, "missing_country": float(scorecard.loc[2, "giá trị"]), "missing_lang": float(scorecard.loc[5, "giá trị"]),
    "non_en_known": float(scorecard.loc[14, "giá trị"]), "missing_abstract": float(scorecard.loc[1, "giá trị"]),
}
INSIGHTS["volume"] = VOLUME.to_dict("records")
INSIGHTS["vn"] = VN_INSIGHT
json.dump(INSIGHTS, open(f"{OUT}/insights.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2,
          default=lambda o: o.item())  # o.item(): đổi số numpy sang số Python
json.dump(MANIFEST, open(f"{FIG}/manifest.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(len(MANIFEST), "biểu đồ đã lưu")
{k: v for k, v in INSIGHTS.items() if k not in ("boot", "yoy", "theme_papers", "filter", "volume", "vn")}

16 biểu đồ đã lưu


{'download_date': '2026-10-23',
 'api_count': 23712,
 'n_clean': 23540,
 'n_countries': 165,
 'n_institutions': 400,
 'n_edges': 70,
 'min_coauth': 10,
 'precision': 96,
 'precision_margin': np.float64(3.8),
 'precision_k': 100,
 'count_2022': 218,
 'count_2025': 8078,
 'count_2026': 10175,
 'mult_2025_2022': 37.1,
 'share_2023_on': 97.1,
 'cagr_2018_2022': 57,
 'cagr_2022_2025': 233,
 'yoy_std': 134,
 'rolling_2022_12': 22,
 'rolling_last': 920,
 'last_month': '09/2026',
 'asia_2019': 43.4,
 'asia_2026': 53.3,
 'europe_2019': 30.2,
 'europe_2026': 19.9,
 'na_2019': 13.2,
 'na_2026': 10.9,
 'top3_2025': [['Trung Quốc', 822], ['Indonesia', 613], ['Hoa Kỳ', 479]],
 'vn_2025': 92,
 'vn_2022': 1,
 'vn_rank_2025': 21,
 'cited_pre': 71.8,
 'cited_post': 34.4,
 'top_first': ['educational technology', 'chatbots'],
 'top_last': ['generative AI', 'ChatGPT'],
 'top_last_share': 26.3,
 'vn_total': 285,
 'intl_share': np.float64(15.0),
 'hub': 'Trung Quốc',
 'hub_degree': 14,
 'genai_share': np.flo